In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
SRC = "/content/drive/MyDrive/tagging/test.h5"
# Target path on the Colab temporary disk
DST = "/content/test.h5"

!rsync -ah --info=progress2 "$SRC" "$DST"

In [ ]:
import os
import sys
import importlib
import re
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from IPython.display import display
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split


# Path setup: in Colab, ROOT_DIR is usually /content, and data is synced to /content/test.h5 first.
ROOT_DIR = os.getcwd()
DATA_DIR = ROOT_DIR
THIS_DIR = f'{ROOT_DIR}/drive/MyDrive/tagging/unsmear/joint_no_fusion'
MODULE_DIR = THIS_DIR
UNSMEAR_DIR = os.path.abspath(os.path.join(MODULE_DIR, '..'))
TAGGING_DIR = os.path.abspath(os.path.join(UNSMEAR_DIR, '..'))
DATA_PATH = os.path.join(DATA_DIR, 'test.h5')

TOOL_DIR = os.path.join(MODULE_DIR, 'tool')
sys.path.insert(0, TOOL_DIR)

import io_utils as io_utils_mod  # noqa: E402
import preprocessing as preprocessing_mod  # noqa: E402
import datasets as datasets_mod  # noqa: E402
import metrics as metrics_mod  # noqa: E402
import training as training_mod  # noqa: E402
import analysis as analysis_mod  # noqa: E402
import model as model_mod  # noqa: E402

for _mod in (io_utils_mod, preprocessing_mod, datasets_mod, metrics_mod, training_mod, analysis_mod, model_mod):
    importlib.reload(_mod)

from io_utils import (  # noqa: E402
    ensure_dir,
    find_existing_repeat_dir,
    load_checkpoint,
    load_prediction_bundle,
    repeat_artifact_paths,
    save_config,
    save_prediction_bundle,
    save_rows_csv,
    set_repeat_seed,
)
from preprocessing import (  # noqa: E402
    HLTEffectsCfg,
    apply_hlt_effects_pair,
    compute_features_with_axis,
    compute_jet_axis,
    get_feat_names,
    get_stats,
    standardize,
    wrap_dphi_np,
)
from datasets import (  # noqa: E402
    JetDataset,
    JointJetDataset,
    make_epoch_hlt_train_loader as build_epoch_hlt_train_loader,
    make_epoch_joint_train_loader as build_epoch_joint_train_loader,
)
from metrics import (  # noqa: E402
    compute_roc,
    fpr_at_target_tpr,
    gap_recovery,
    maybe_wrap_residual,
    metric_dict,
    predict_joint_reco,
)
from training import (  # noqa: E402
    eval_joint_model,
    eval_kd_student,
    evaluate,
    train_or_load_joint_model,
    train_or_load_kd_standard_model,
    train_or_load_standard_model,
)
from analysis import (  # noqa: E402
    build_binned_curve,
    collect_case_rows,
    collect_distance_logit_rows,
    collect_embedding_distance_rows,
    corr_safe,
    extract_teacher_embedding,
    per_sample_embedding_mse,
)
from model import ParticleTransformerKD, SharedEncoderUnsmearClassifier  # noqa: E402

seed = 42
np.random.seed(seed)
torch.manual_seed(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

RUN_NAME = 'unsmear_transformer_sharedencoder_no_fusion_repeat3'
OUT_DIR = os.path.join(MODULE_DIR, 'runs', RUN_NAME)
FIG_DIR = os.path.join(OUT_DIR, 'figs')
CKPT_DIR = os.path.join(OUT_DIR, 'ckpts')
METRICS_DIR = os.path.join(OUT_DIR, 'metrics')
REPEAT_DIR = os.path.join(OUT_DIR, 'repeats')
TABLE_DIR = os.path.join(OUT_DIR, 'tables')
GRAD_PROBE_DIR = os.path.join(OUT_DIR, 'grad_probe')

ensure_dir(FIG_DIR)
ensure_dir(CKPT_DIR)
ensure_dir(METRICS_DIR)
ensure_dir(REPEAT_DIR)
ensure_dir(TABLE_DIR)
ensure_dir(GRAD_PROBE_DIR)

CONFIG = {
    'data_path': DATA_PATH,
    'n_jets': 2000000,
    'max_particles': 100,
    'feature_kind': '7d',
    'repeat_seeds': [42],
    'load_shared_baselines': False,
    'load_joint_model': False,
    'shared_backbone': {
        'input_dim': 7,
        'embed_dim': 128,
        'num_heads': 8,
        'num_layers': 6,
        'ff_dim': 512,
        'dropout': 0.1,
        'add_mask_channel': False,
        'use_positional_embedding': False,
        'max_seq_len': 128,
        'mask_encoder_input': False,
        'cls_pool_heads': 4,
    },
    'tagger': {},
    'joint_model': {
        'unsmear_decoder_layers': 2,
        'unsmear_decoder_heads': 8,
        'unsmear_decoder_ff_dim': 512,
        'unsmear_decoder_dropout': 0.1,
        'return_reco': True,
        'mask_output': True,
        'cls_use_delta_fusion': False,
        'cls_detach_delta_for_cls': True,
        'cls_gate_hidden_dim': 128,
        'cls_gate_init_bias': -2.0,
        'cls_alpha_init': 0.05,
    },
    'hlt_effects': {
        'pt_threshold_offline': 0,
        'pt_threshold_hlt': 0,
        'pt_resolution': 0.10,
        'eta_resolution': 0.03,
        'phi_resolution': 0.03,
    },
    'kd': {
        'enable': True,
        'temperature': 2.0,
        'alpha_kd': 0.5,
        'alpha_attn': 0,
    },
    'training': {
        'batch_size': 2048,
        'epochs': 70,
        'lr': 5e-4,
        'weight_decay': 1e-5,
        'warmup_epochs': 5,
        'patience': 8,
        'early_stop_metric': 'val_auc',
        'use_sample_weight_for_all_losses': False,
        'joint_unsmear_weight': 1.6,
        'joint_cls_weight': 0.8,
        'joint_phys_weight': 0.0,
        'feature_loss_weights': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
        'resmear_each_epoch_baselines': True,
        'resmear_each_epoch_joint': True,
        'resmear_seed_stride': 1,
    },
    'grad_probe': {
        'train_batches_per_epoch': 100,
        'val_batches_per_epoch': 20,
    },
}

feat_names = get_feat_names(CONFIG['feature_kind'])
shared_backbone_cfg = dict(CONFIG['shared_backbone'])
shared_backbone_cfg['input_dim'] = len(feat_names)
shared_backbone_cfg['max_seq_len'] = int(CONFIG['max_particles'])
CONFIG['shared_backbone'] = shared_backbone_cfg
CONFIG['tagger'] = dict(shared_backbone_cfg)
CONFIG['joint_model'] = {**shared_backbone_cfg, **dict(CONFIG['joint_model'])}

feature_loss_weights = np.asarray(CONFIG['training']['feature_loss_weights'], dtype=np.float32)
if feature_loss_weights.shape[0] != len(feat_names):
    raise ValueError(
        f"Expected {len(feat_names)} feature weights for {CONFIG['feature_kind']}, got {feature_loss_weights.shape[0]}"
    )
CONFIG['training']['feature_loss_weights'] = feature_loss_weights.tolist()
CONFIG['training']['use_sample_weight_for_all_losses'] = bool(
    CONFIG['training'].get('use_sample_weight_for_all_losses', True)
)

config_path = os.path.join(OUT_DIR, 'config.json')
save_config(CONFIG, config_path)
print('Data path:', CONFIG['data_path'])
print('Run dir:', OUT_DIR)
print('Feature kind:', CONFIG['feature_kind'], 'feat_names:', feat_names)
print('Repeat seeds:', CONFIG['repeat_seeds'])
print('Shared backbone cfg:', CONFIG['shared_backbone'])
print('Feature loss weights:', dict(zip(feat_names, np.round(feature_loss_weights, 4))))
print('Joint physical consistency weight:', float(CONFIG['training']['joint_phys_weight']))
print('Use sample weight for all losses:', bool(CONFIG['training']['use_sample_weight_for_all_losses']))
print('Delta fusion enabled:', bool(CONFIG['joint_model']['cls_use_delta_fusion']))
print('Detach delta for classifier:', bool(CONFIG['joint_model']['cls_detach_delta_for_cls']))
print('Gradient probe cfg:', CONFIG['grad_probe'])


In [ ]:
# Load the raw constituents and build the offline / HLT views
n = int(CONFIG['n_jets'])
S = int(CONFIG['max_particles'])

with h5py.File(CONFIG['data_path'], 'r') as f:
    labels = f['labels'][:n].astype(np.int64)
    weights = f['weights'][:n].astype(np.float32)
    pt = f['fjet_clus_pt'][:n, :S].astype(np.float32)
    eta = f['fjet_clus_eta'][:n, :S].astype(np.float32)
    phi = f['fjet_clus_phi'][:n, :S].astype(np.float32)
    E = f['fjet_clus_E'][:n, :S].astype(np.float32)

constituents_raw = np.stack([pt, eta, phi, E], axis=-1)
mask_raw = pt > 0
print('Raw:', constituents_raw.shape, 'mask:', mask_raw.shape)
print('Signal:', int(labels.sum()), 'Bkg:', int((1 - labels).sum()))

hcfg = HLTEffectsCfg(**CONFIG['hlt_effects'])
_, hlt_const, hlt_mask = apply_hlt_effects_pair(
    constituents_raw,
    mask_raw,
    hcfg,
    seed=seed,
)

pt_thr_off = float(CONFIG['hlt_effects']['pt_threshold_offline'])
off_mask = mask_raw & (constituents_raw[:, :, 0] >= pt_thr_off)
off_const = constituents_raw.copy()
off_const[~off_mask] = 0.0
hlt_const = hlt_const.copy()
hlt_const[~hlt_mask] = 0.0

axis_off = compute_jet_axis(off_const, off_mask)
axis_hlt = compute_jet_axis(hlt_const, hlt_mask)
feat_off = compute_features_with_axis(off_const, off_mask, axis_off, kind=CONFIG['feature_kind'])
feat_hlt = compute_features_with_axis(hlt_const, hlt_mask, axis_hlt, kind=CONFIG['feature_kind'])

idx = np.arange(len(labels))
train_idx, temp_idx = train_test_split(idx, test_size=1/3, random_state=seed, stratify=labels)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, random_state=seed, stratify=labels[temp_idx])
print(f'Split: train={len(train_idx):,} val={len(val_idx):,} test={len(test_idx):,}')

feat_means, feat_stds = get_stats(feat_off, off_mask, train_idx)
feat_off_std = standardize(feat_off, off_mask, feat_means, feat_stds, clip=10.0)
feat_hlt_std = standardize(feat_hlt, hlt_mask, feat_means, feat_stds, clip=10.0)
common_mask = off_mask & hlt_mask

x_joint = feat_hlt_std.copy()
y_joint = feat_off_std.copy()
x_joint[~common_mask] = 0.0
y_joint[~common_mask] = 0.0

train_const_raw = constituents_raw[train_idx]
train_mask_raw = mask_raw[train_idx]

print('Offline/HLT feature shape:', feat_off_std.shape, feat_hlt_std.shape)
print('Mask identical:', bool(np.array_equal(off_mask, hlt_mask)))
print('Common-mask fraction:', float(common_mask.mean()))
print('Feat means:', np.round(feat_means, 4))
print('Feat stds :', np.round(feat_stds, 4))
print('Baseline epoch resmear enabled:', bool(CONFIG['training'].get('resmear_each_epoch_baselines', False)))
print('Joint epoch resmear enabled:', bool(CONFIG['training'].get('resmear_each_epoch_joint', False)))


In [ ]:
# Build the datasets and loaders
BS = int(CONFIG['training']['batch_size'])
train_ds_hlt = JetDataset(
    feat_off_std[train_idx],
    feat_hlt_std[train_idx],
    labels[train_idx],
    off_mask[train_idx],
    hlt_mask[train_idx],
    weights[train_idx],
)
val_ds_hlt = JetDataset(
    feat_off_std[val_idx],
    feat_hlt_std[val_idx],
    labels[val_idx],
    off_mask[val_idx],
    hlt_mask[val_idx],
    weights[val_idx],
)
test_ds_hlt = JetDataset(
    feat_off_std[test_idx],
    feat_hlt_std[test_idx],
    labels[test_idx],
    off_mask[test_idx],
    hlt_mask[test_idx],
    weights[test_idx],
)

train_ds_joint = JointJetDataset(
    x_joint[train_idx],
    y_joint[train_idx],
    common_mask[train_idx],
    labels[train_idx],
    weights[train_idx],
)
val_ds_joint = JointJetDataset(
    x_joint[val_idx],
    y_joint[val_idx],
    common_mask[val_idx],
    labels[val_idx],
    weights[val_idx],
)
test_ds_joint = JointJetDataset(
    x_joint[test_idx],
    y_joint[test_idx],
    common_mask[test_idx],
    labels[test_idx],
    weights[test_idx],
)

train_loader_hlt = DataLoader(train_ds_hlt, batch_size=BS, shuffle=True, drop_last=True)
val_loader_hlt = DataLoader(val_ds_hlt, batch_size=BS, shuffle=False)
test_loader_hlt = DataLoader(test_ds_hlt, batch_size=BS, shuffle=False)

train_loader_joint = DataLoader(train_ds_joint, batch_size=BS, shuffle=True, drop_last=True)
val_loader_joint = DataLoader(val_ds_joint, batch_size=BS, shuffle=False)
test_loader_joint = DataLoader(test_ds_joint, batch_size=BS, shuffle=False)



In [ ]:
# Run all models with multiple seeds and save each repeat checkpoint, metrics, and predictions.
train_cfg = CONFIG['training']
kd_cfg = CONFIG['kd']
grad_probe_cfg = CONFIG.get('grad_probe', {})
use_sample_weight_for_all_losses = bool(train_cfg.get('use_sample_weight_for_all_losses', True))
joint_feature_loss_weights = np.asarray(train_cfg['feature_loss_weights'], dtype=np.float32)
REPEAT_SEEDS = [int(x) for x in CONFIG.get('repeat_seeds', [42, 52, 62])]


def set_repeat_seed(seed_value: int):
    np.random.seed(int(seed_value))
    torch.manual_seed(int(seed_value))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed_value))


def _make_repeat_hlt_train_loader(epoch: int, repeat_seed: int):
    return build_epoch_hlt_train_loader(
        epoch=int(epoch),
        batch_size=BS,
        feat_off_train=feat_off_std[train_idx],
        off_mask_train=off_mask[train_idx],
        labels_train=labels[train_idx],
        weights_train=weights[train_idx],
        train_const_raw=train_const_raw,
        train_mask_raw=train_mask_raw,
        cfg=hcfg,
        feature_kind=CONFIG['feature_kind'],
        means=feat_means,
        stds=feat_stds,
        seed=int(repeat_seed),
        fixed_feat_hlt_train=feat_hlt_std[train_idx],
        fixed_hlt_mask_train=hlt_mask[train_idx],
        seed_stride=int(CONFIG['training'].get('resmear_seed_stride', 1)),
        resmear_each_epoch=bool(CONFIG['training'].get('resmear_each_epoch_baselines', False)),
        clip=10.0,
    )


def _make_repeat_joint_train_loader(epoch: int, repeat_seed: int):
    return build_epoch_joint_train_loader(
        epoch=int(epoch),
        batch_size=BS,
        labels_train=labels[train_idx],
        weights_train=weights[train_idx],
        train_const_raw=train_const_raw,
        train_mask_raw=train_mask_raw,
        cfg=hcfg,
        feature_kind=CONFIG['feature_kind'],
        means=feat_means,
        stds=feat_stds,
        seed=int(repeat_seed),
        fixed_x_train=x_joint[train_idx],
        fixed_y_train=y_joint[train_idx],
        fixed_mask_train=common_mask[train_idx],
        seed_stride=int(CONFIG['training'].get('resmear_seed_stride', 1)),
        resmear_each_epoch=bool(CONFIG['training'].get('resmear_each_epoch_joint', True)),
        clip=10.0,
    )

def repeat_artifact_paths(repeat_dir: str):
    repeat_ckpt_dir = os.path.join(repeat_dir, 'ckpts')
    repeat_metrics_dir = os.path.join(repeat_dir, 'metrics')
    repeat_pred_dir = os.path.join(repeat_dir, 'predictions')
    repeat_grad_probe_dir = os.path.join(repeat_dir, 'grad_probe')
    return {
        'repeat_dir': repeat_dir,
        'repeat_ckpt_dir': repeat_ckpt_dir,
        'repeat_metrics_dir': repeat_metrics_dir,
        'repeat_pred_dir': repeat_pred_dir,
        'repeat_grad_probe_dir': repeat_grad_probe_dir,
        'grad_probe_prefixes': {
            'joint_no_kd': os.path.join(repeat_grad_probe_dir, 'joint_no_kd'),
            'joint_with_kd': os.path.join(repeat_grad_probe_dir, 'joint_with_kd'),
        },
        'epoch_metrics_paths': {
            'teacher_off': os.path.join(repeat_metrics_dir, 'teacher_off_epoch_metrics.csv'),
            'student_hlt': os.path.join(repeat_metrics_dir, 'student_hlt_epoch_metrics.csv'),
            'hlt_kd': os.path.join(repeat_metrics_dir, 'hlt_kd_epoch_metrics.csv'),
            'joint_no_kd': os.path.join(repeat_metrics_dir, 'joint_no_kd_epoch_metrics.csv'),
            'joint_with_kd': os.path.join(repeat_metrics_dir, 'joint_with_kd_epoch_metrics.csv'),
        },
        'ckpt_paths': {
            'Teacher(OFF_FULL)': os.path.join(repeat_ckpt_dir, 'teacher_offline.pt'),
            'Student(HLT)': os.path.join(repeat_ckpt_dir, 'student_hlt.pt'),
            'Student(HLT)+KD': os.path.join(repeat_ckpt_dir, 'student_hlt_kd.pt'),
            'JointSharedEncoder(HLT,no_kd)': os.path.join(repeat_ckpt_dir, 'joint_sharedencoder_no_kd.pt'),
            'JointSharedEncoder(HLT,with_kd)': os.path.join(repeat_ckpt_dir, 'joint_sharedencoder_with_kd.pt'),
        },
        'prediction_paths': {
            'Teacher(OFF_FULL)': os.path.join(repeat_pred_dir, 'teacher_offline_test_preds.npz'),
            'Student(HLT)': os.path.join(repeat_pred_dir, 'student_hlt_test_preds.npz'),
            'Student(HLT)+KD': os.path.join(repeat_pred_dir, 'student_hlt_kd_test_preds.npz'),
            'JointSharedEncoder(HLT,no_kd)': os.path.join(repeat_pred_dir, 'joint_sharedencoder_no_kd_test_preds.npz'),
            'JointSharedEncoder(HLT,with_kd)': os.path.join(repeat_pred_dir, 'joint_sharedencoder_with_kd_test_preds.npz'),
        },
    }


def load_prediction_bundle(path: str):
    data = np.load(path)
    return {
        'preds': np.asarray(data['preds'], dtype=np.float32),
        'labels': np.asarray(data['labels'], dtype=np.float32),
        'weights': np.asarray(data['weights'], dtype=np.float32),
    }


def _load_existing_repeat_result(repeat_idx: int, repeat_seed: int):
    repeat_dir = find_existing_repeat_dir(REPEAT_DIR, int(repeat_seed))
    if repeat_dir is None:
        return None

    artifacts = repeat_artifact_paths(repeat_dir)
    required_paths = []
    required_paths.extend(list(artifacts['ckpt_paths'].values()))
    required_paths.extend(list(artifacts['prediction_paths'].values()))
    required_paths.extend(list(artifacts['epoch_metrics_paths'].values()))
    if int(grad_probe_cfg.get('train_batches_per_epoch', 0)) > 0 or int(grad_probe_cfg.get('val_batches_per_epoch', 0)) > 0:
        for probe_prefix in artifacts['grad_probe_prefixes'].values():
            required_paths.extend([
                f'{probe_prefix}_scalar_losses.csv',
                f'{probe_prefix}_grad_norms.csv',
                f'{probe_prefix}_grad_cosines.csv',
            ])
    missing_paths = [p for p in required_paths if not os.path.isfile(p)]
    if missing_paths:
        print(f'Seed {repeat_seed}: found existing folder but artifacts are incomplete, retrain this seed.')
        for p in missing_paths[:5]:
            print('  missing:', p)
        if len(missing_paths) > 5:
            print(f'  ... and {len(missing_paths) - 5} more missing files')
        return None

    model_to_metric_key = {
        'Teacher(OFF_FULL)': 'teacher_off',
        'Student(HLT)': 'student_hlt',
        'Student(HLT)+KD': 'hlt_kd',
        'JointSharedEncoder(HLT,no_kd)': 'joint_no_kd',
        'JointSharedEncoder(HLT,with_kd)': 'joint_with_kd',
    }

    models = {}
    for model_name, pred_path in artifacts['prediction_paths'].items():
        bundle = load_prediction_bundle(pred_path)
        sample_weight = np.asarray(bundle['weights'], dtype=np.float64) if use_sample_weight_for_all_losses else None
        _fpr, _tpr, auc, auc_weighted = compute_roc(bundle['labels'], bundle['preds'], sample_weight=sample_weight)
        metric_key = model_to_metric_key[model_name]
        models[model_name] = {
            'auc': float(auc),
            'auc_weighted': float(auc_weighted),
            'preds': bundle['preds'],
            'labels': bundle['labels'],
            'weights': bundle['weights'],
            'ckpt_path': artifacts['ckpt_paths'][model_name],
            'prediction_path': pred_path,
            'epoch_metrics_path': artifacts['epoch_metrics_paths'][metric_key],
            'grad_probe_prefix': artifacts['grad_probe_prefixes'].get(metric_key),
        }

    print(f'Seed {repeat_seed}: found existing repeat folder, load and skip training -> {repeat_dir}')
    return {
        'repeat_idx': int(repeat_idx),
        'repeat_seed': int(repeat_seed),
        'repeat_dir': repeat_dir,
        'models': models,
    }


repeat_results = []
print('Repeat seeds:', REPEAT_SEEDS)

for repeat_idx, repeat_seed in enumerate(REPEAT_SEEDS):
    print()
    print(f'==== Repeat {repeat_idx + 1}/{len(REPEAT_SEEDS)} seed={repeat_seed} ====')
    cached_repeat_result = _load_existing_repeat_result(int(repeat_idx), int(repeat_seed))
    if cached_repeat_result is not None:
        repeat_results.append(cached_repeat_result)
        continue

    set_repeat_seed(int(repeat_seed))

    repeat_dir = os.path.join(REPEAT_DIR, f'repeat_{int(repeat_idx):02d}_seed_{int(repeat_seed)}')
    repeat_ckpt_dir = os.path.join(repeat_dir, 'ckpts')
    repeat_metrics_dir = os.path.join(repeat_dir, 'metrics')
    repeat_pred_dir = os.path.join(repeat_dir, 'predictions')
    repeat_grad_probe_dir = os.path.join(repeat_dir, 'grad_probe')
    ensure_dir(repeat_dir)
    ensure_dir(repeat_ckpt_dir)
    ensure_dir(repeat_metrics_dir)
    ensure_dir(repeat_pred_dir)
    ensure_dir(repeat_grad_probe_dir)

    epoch_metrics_paths = {
        'teacher_off': os.path.join(repeat_metrics_dir, 'teacher_off_epoch_metrics.csv'),
        'student_hlt': os.path.join(repeat_metrics_dir, 'student_hlt_epoch_metrics.csv'),
        'hlt_kd': os.path.join(repeat_metrics_dir, 'hlt_kd_epoch_metrics.csv'),
        'joint_no_kd': os.path.join(repeat_metrics_dir, 'joint_no_kd_epoch_metrics.csv'),
        'joint_with_kd': os.path.join(repeat_metrics_dir, 'joint_with_kd_epoch_metrics.csv'),
    }
    grad_probe_joint_no_kd = {
        'model_name': 'joint_no_kd',
        'output_prefix': os.path.join(repeat_grad_probe_dir, 'joint_no_kd'),
        'train_batches_per_epoch': int(grad_probe_cfg.get('train_batches_per_epoch', 100)),
        'val_batches_per_epoch': int(grad_probe_cfg.get('val_batches_per_epoch', 20)),
    }
    grad_probe_joint_with_kd = {
        'model_name': 'joint_with_kd',
        'output_prefix': os.path.join(repeat_grad_probe_dir, 'joint_with_kd'),
        'train_batches_per_epoch': int(grad_probe_cfg.get('train_batches_per_epoch', 100)),
        'val_batches_per_epoch': int(grad_probe_cfg.get('val_batches_per_epoch', 20)),
    }

    hlt_train_loader_factory = (
        (lambda epoch, repeat_seed=repeat_seed: _make_repeat_hlt_train_loader(epoch, int(repeat_seed)))
        if bool(train_cfg.get('resmear_each_epoch_baselines', False))
        else None
    )
    joint_train_loader_factory = (
        (lambda epoch, repeat_seed=repeat_seed: _make_repeat_joint_train_loader(epoch, int(repeat_seed)))
        if bool(train_cfg.get('resmear_each_epoch_joint', True))
        else None
    )

    teacher = ParticleTransformerKD(**CONFIG['tagger']).to(device)
    teacher_ckpt = os.path.join(repeat_ckpt_dir, 'teacher_offline.pt')
    teacher = train_or_load_standard_model(
        'Teacher(OFF_FULL)',
        teacher,
        teacher_ckpt,
        train_loader_hlt,
        val_loader_hlt,
        device=device,
        feat_key='off',
        mask_key='mask_off',
        allow_load=bool(CONFIG.get('load_shared_baselines', False)),
        lr=float(train_cfg['lr']),
        weight_decay=float(train_cfg['weight_decay']),
        warmup_epochs=int(train_cfg['warmup_epochs']),
        epochs=int(train_cfg['epochs']),
        patience=int(train_cfg['patience']),
        early_stop_metric=str(train_cfg['early_stop_metric']),
        use_sample_weight_for_all_losses=use_sample_weight_for_all_losses,
        epoch_metrics_path=epoch_metrics_paths['teacher_off'],
    )
    auc_teacher, auc_teacher_w, p_teacher, y_true, w_true = evaluate(
        teacher,
        test_loader_hlt,
        device,
        'off',
        'mask_off',
        use_sample_weight_for_all_losses=use_sample_weight_for_all_losses,
    )
    teacher_pred_path = os.path.join(repeat_pred_dir, 'teacher_offline_test_preds.npz')
    save_prediction_bundle(teacher_pred_path, preds=p_teacher, labels=y_true, weights=w_true)

    student_hlt = ParticleTransformerKD(**CONFIG['tagger']).to(device)
    student_hlt_ckpt = os.path.join(repeat_ckpt_dir, 'student_hlt.pt')
    student_hlt = train_or_load_standard_model(
        'Student(HLT)',
        student_hlt,
        student_hlt_ckpt,
        train_loader_hlt,
        val_loader_hlt,
        device=device,
        feat_key='hlt',
        mask_key='mask_hlt',
        allow_load=bool(CONFIG.get('load_shared_baselines', False)),
        lr=float(train_cfg['lr']),
        weight_decay=float(train_cfg['weight_decay']),
        warmup_epochs=int(train_cfg['warmup_epochs']),
        epochs=int(train_cfg['epochs']),
        patience=int(train_cfg['patience']),
        early_stop_metric=str(train_cfg['early_stop_metric']),
        use_sample_weight_for_all_losses=use_sample_weight_for_all_losses,
        train_loader_factory=hlt_train_loader_factory,
        epoch_metrics_path=epoch_metrics_paths['student_hlt'],
    )
    auc_hlt, auc_hlt_w, p_hlt, _, _ = evaluate(
        student_hlt,
        test_loader_hlt,
        device,
        'hlt',
        'mask_hlt',
        use_sample_weight_for_all_losses=use_sample_weight_for_all_losses,
    )
    hlt_pred_path = os.path.join(repeat_pred_dir, 'student_hlt_test_preds.npz')
    save_prediction_bundle(hlt_pred_path, preds=p_hlt, labels=y_true, weights=w_true)

    student_hlt_kd = ParticleTransformerKD(**CONFIG['tagger']).to(device)
    student_hlt_kd_ckpt = os.path.join(repeat_ckpt_dir, 'student_hlt_kd.pt')
    student_hlt_kd = train_or_load_kd_standard_model(
        'Student(HLT)+KD',
        student_hlt_kd,
        teacher,
        student_hlt_kd_ckpt,
        train_loader_hlt,
        val_loader_hlt,
        device=device,
        allow_load=bool(CONFIG.get('load_shared_baselines', False)),
        lr=float(train_cfg['lr']),
        weight_decay=float(train_cfg['weight_decay']),
        warmup_epochs=int(train_cfg['warmup_epochs']),
        epochs=int(train_cfg['epochs']),
        patience=int(train_cfg['patience']),
        early_stop_metric=str(train_cfg['early_stop_metric']),
        use_sample_weight_for_all_losses=use_sample_weight_for_all_losses,
        kd_temperature=float(kd_cfg['temperature']),
        kd_alpha=float(kd_cfg['alpha_kd']),
        kd_alpha_attn=float(kd_cfg['alpha_attn']),
        train_loader_factory=hlt_train_loader_factory,
        epoch_metrics_path=epoch_metrics_paths['hlt_kd'],
    )
    hlt_kd_test = eval_kd_student(
        student_hlt_kd,
        teacher,
        test_loader_hlt,
        device,
        {'kd': {'temperature': float(kd_cfg['temperature']), 'alpha_kd': float(kd_cfg['alpha_kd']), 'alpha_attn': float(kd_cfg['alpha_attn'])}},
        use_sample_weight_for_all_losses=use_sample_weight_for_all_losses,
    )
    hlt_kd_pred_path = os.path.join(repeat_pred_dir, 'student_hlt_kd_test_preds.npz')
    save_prediction_bundle(
        hlt_kd_pred_path,
        preds=np.asarray(hlt_kd_test['preds'], dtype=np.float32),
        labels=np.asarray(hlt_kd_test['labels'], dtype=np.float32),
        weights=np.asarray(hlt_kd_test['weights'], dtype=np.float32),
    )

    joint_model_no_kd = SharedEncoderUnsmearClassifier(**CONFIG['joint_model']).to(device)
    joint_ckpt_no_kd = os.path.join(repeat_ckpt_dir, 'joint_sharedencoder_no_kd.pt')
    joint_model_no_kd = train_or_load_joint_model(
        'JointSharedEncoder(HLT,no_kd)',
        joint_model_no_kd,
        joint_ckpt_no_kd,
        train_loader_joint,
        val_loader_joint,
        device=device,
        feat_names=feat_names,
        feat_means=feat_means,
        feat_stds=feat_stds,
        feature_loss_weights=joint_feature_loss_weights,
        joint_phys_weight=float(train_cfg['joint_phys_weight']),
        joint_unsmear_weight=float(train_cfg['joint_unsmear_weight']),
        joint_cls_weight=float(train_cfg['joint_cls_weight']),
        lr=float(train_cfg['lr']),
        weight_decay=float(train_cfg['weight_decay']),
        warmup_epochs=int(train_cfg['warmup_epochs']),
        epochs=int(train_cfg['epochs']),
        patience=int(train_cfg['patience']),
        early_stop_metric=str(train_cfg['early_stop_metric']),
        use_sample_weight_for_all_losses=use_sample_weight_for_all_losses,
        teacher=teacher,
        use_kd=False,
        kd_temperature=float(kd_cfg['temperature']),
        kd_alpha=float(kd_cfg['alpha_kd']),
        kd_alpha_attn=float(kd_cfg['alpha_attn']),
        allow_load=bool(CONFIG.get('load_joint_model', False)),
        train_loader_factory=joint_train_loader_factory,
        grad_probe_cfg=grad_probe_joint_no_kd,
        epoch_metrics_path=epoch_metrics_paths['joint_no_kd'],
    )
    joint_test_no_kd = eval_joint_model(
        joint_model_no_kd,
        test_loader_joint,
        device=device,
        feat_names=feat_names,
        feat_means=feat_means,
        feat_stds=feat_stds,
        feature_loss_weights=joint_feature_loss_weights,
        joint_phys_weight=float(train_cfg['joint_phys_weight']),
        joint_unsmear_weight=float(train_cfg['joint_unsmear_weight']),
        joint_cls_weight=float(train_cfg['joint_cls_weight']),
        teacher=teacher,
        use_kd=False,
        kd_temperature=float(kd_cfg['temperature']),
        kd_alpha=float(kd_cfg['alpha_kd']),
        kd_alpha_attn=float(kd_cfg['alpha_attn']),
        use_sample_weight_for_all_losses=use_sample_weight_for_all_losses,
    )
    joint_no_kd_pred_path = os.path.join(repeat_pred_dir, 'joint_sharedencoder_no_kd_test_preds.npz')
    save_prediction_bundle(
        joint_no_kd_pred_path,
        preds=np.asarray(joint_test_no_kd['preds'], dtype=np.float32),
        labels=np.asarray(joint_test_no_kd['labels'], dtype=np.float32),
        weights=np.asarray(joint_test_no_kd['weights'], dtype=np.float32),
    )

    joint_model_with_kd = SharedEncoderUnsmearClassifier(**CONFIG['joint_model']).to(device)
    joint_ckpt_with_kd = os.path.join(repeat_ckpt_dir, 'joint_sharedencoder_with_kd.pt')
    joint_model_with_kd = train_or_load_joint_model(
        'JointSharedEncoder(HLT,with_kd)',
        joint_model_with_kd,
        joint_ckpt_with_kd,
        train_loader_joint,
        val_loader_joint,
        device=device,
        feat_names=feat_names,
        feat_means=feat_means,
        feat_stds=feat_stds,
        feature_loss_weights=joint_feature_loss_weights,
        joint_phys_weight=float(train_cfg['joint_phys_weight']),
        joint_unsmear_weight=float(train_cfg['joint_unsmear_weight']),
        joint_cls_weight=float(train_cfg['joint_cls_weight']),
        lr=float(train_cfg['lr']),
        weight_decay=float(train_cfg['weight_decay']),
        warmup_epochs=int(train_cfg['warmup_epochs']),
        epochs=int(train_cfg['epochs']),
        patience=int(train_cfg['patience']),
        early_stop_metric=str(train_cfg['early_stop_metric']),
        use_sample_weight_for_all_losses=use_sample_weight_for_all_losses,
        teacher=teacher,
        use_kd=True,
        kd_temperature=float(kd_cfg['temperature']),
        kd_alpha=float(kd_cfg['alpha_kd']),
        kd_alpha_attn=float(kd_cfg['alpha_attn']),
        allow_load=bool(CONFIG.get('load_joint_model', False)),
        train_loader_factory=joint_train_loader_factory,
        grad_probe_cfg=grad_probe_joint_with_kd,
        epoch_metrics_path=epoch_metrics_paths['joint_with_kd'],
    )
    joint_test_with_kd = eval_joint_model(
        joint_model_with_kd,
        test_loader_joint,
        device=device,
        feat_names=feat_names,
        feat_means=feat_means,
        feat_stds=feat_stds,
        feature_loss_weights=joint_feature_loss_weights,
        joint_phys_weight=float(train_cfg['joint_phys_weight']),
        joint_unsmear_weight=float(train_cfg['joint_unsmear_weight']),
        joint_cls_weight=float(train_cfg['joint_cls_weight']),
        teacher=teacher,
        use_kd=True,
        kd_temperature=float(kd_cfg['temperature']),
        kd_alpha=float(kd_cfg['alpha_kd']),
        kd_alpha_attn=float(kd_cfg['alpha_attn']),
        use_sample_weight_for_all_losses=use_sample_weight_for_all_losses,
    )
    joint_with_kd_pred_path = os.path.join(repeat_pred_dir, 'joint_sharedencoder_with_kd_test_preds.npz')
    save_prediction_bundle(
        joint_with_kd_pred_path,
        preds=np.asarray(joint_test_with_kd['preds'], dtype=np.float32),
        labels=np.asarray(joint_test_with_kd['labels'], dtype=np.float32),
        weights=np.asarray(joint_test_with_kd['weights'], dtype=np.float32),
    )

    repeat_results.append({
        'repeat_idx': int(repeat_idx),
        'repeat_seed': int(repeat_seed),
        'repeat_dir': repeat_dir,
        'models': {
            'Teacher(OFF_FULL)': {
                'auc': float(auc_teacher),
                'auc_weighted': float(auc_teacher_w),
                'preds': np.asarray(p_teacher, dtype=np.float32),
                'labels': np.asarray(y_true, dtype=np.float32),
                'weights': np.asarray(w_true, dtype=np.float32),
                'ckpt_path': teacher_ckpt,
                'prediction_path': teacher_pred_path,
                'epoch_metrics_path': epoch_metrics_paths['teacher_off'],
            },
            'Student(HLT)': {
                'auc': float(auc_hlt),
                'auc_weighted': float(auc_hlt_w),
                'preds': np.asarray(p_hlt, dtype=np.float32),
                'labels': np.asarray(y_true, dtype=np.float32),
                'weights': np.asarray(w_true, dtype=np.float32),
                'ckpt_path': student_hlt_ckpt,
                'prediction_path': hlt_pred_path,
                'epoch_metrics_path': epoch_metrics_paths['student_hlt'],
            },
            'Student(HLT)+KD': {
                'auc': float(hlt_kd_test['auc']),
                'auc_weighted': float(hlt_kd_test['auc_weighted']),
                'preds': np.asarray(hlt_kd_test['preds'], dtype=np.float32),
                'labels': np.asarray(hlt_kd_test['labels'], dtype=np.float32),
                'weights': np.asarray(hlt_kd_test['weights'], dtype=np.float32),
                'ckpt_path': student_hlt_kd_ckpt,
                'prediction_path': hlt_kd_pred_path,
                'epoch_metrics_path': epoch_metrics_paths['hlt_kd'],
                'test_metrics': hlt_kd_test,
            },
            'JointSharedEncoder(HLT,no_kd)': {
                'auc': float(joint_test_no_kd['auc']),
                'auc_weighted': float(joint_test_no_kd['auc_weighted']),
                'preds': np.asarray(joint_test_no_kd['preds'], dtype=np.float32),
                'labels': np.asarray(joint_test_no_kd['labels'], dtype=np.float32),
                'weights': np.asarray(joint_test_no_kd['weights'], dtype=np.float32),
                'ckpt_path': joint_ckpt_no_kd,
                'prediction_path': joint_no_kd_pred_path,
                'epoch_metrics_path': epoch_metrics_paths['joint_no_kd'],
                'grad_probe_prefix': grad_probe_joint_no_kd['output_prefix'],
                'test_metrics': joint_test_no_kd,
            },
            'JointSharedEncoder(HLT,with_kd)': {
                'auc': float(joint_test_with_kd['auc']),
                'auc_weighted': float(joint_test_with_kd['auc_weighted']),
                'preds': np.asarray(joint_test_with_kd['preds'], dtype=np.float32),
                'labels': np.asarray(joint_test_with_kd['labels'], dtype=np.float32),
                'weights': np.asarray(joint_test_with_kd['weights'], dtype=np.float32),
                'ckpt_path': joint_ckpt_with_kd,
                'prediction_path': joint_with_kd_pred_path,
                'epoch_metrics_path': epoch_metrics_paths['joint_with_kd'],
                'grad_probe_prefix': grad_probe_joint_with_kd['output_prefix'],
                'test_metrics': joint_test_with_kd,
            },
        },
    })

print()
print('Completed repeats:', len(repeat_results))


In [ ]:
# Summarize ROC / AUC / FPR@TPR / Gap Recovery across repeats and report means and variances.

train_cfg = CONFIG['training']

use_sample_weight_for_all_losses = bool(train_cfg.get('use_sample_weight_for_all_losses', True))
MODEL_DISPLAY_ORDER = [
    'Teacher(OFF_FULL)',
    'Student(HLT)',
    'Student(HLT)+KD',
    'JointSharedEncoder(HLT,no_kd)',
    'JointSharedEncoder(HLT,with_kd)',
]
MODEL_COLORS = {
    'Teacher(OFF_FULL)': '#4C78A8',
    'Student(HLT)': '#F58518',
    'Student(HLT)+KD': '#54A24B',
    'JointSharedEncoder(HLT,no_kd)': '#E45756',
    'JointSharedEncoder(HLT,with_kd)': '#72B7B2',
}


def gap_recovery(model_fpr: float, baseline_fpr: float, teacher_fpr: float) -> float:
    denom = float(baseline_fpr - teacher_fpr)
    if abs(denom) < 1e-12:
        return float('nan')
    return float((baseline_fpr - model_fpr) / denom)


repeat_detail_rows = []
for repeat_result in repeat_results:
    repeat_seed = int(repeat_result['repeat_seed'])
    teacher_result = repeat_result['models']['Teacher(OFF_FULL)']
    hlt_result = repeat_result['models']['Student(HLT)']
    teacher_weight = np.asarray(teacher_result['weights'], dtype=np.float64) if use_sample_weight_for_all_losses else None
    hlt_weight = np.asarray(hlt_result['weights'], dtype=np.float64) if use_sample_weight_for_all_losses else None
    teacher_fpr, teacher_tpr, _, _ = compute_roc(teacher_result['labels'], teacher_result['preds'], sample_weight=teacher_weight)
    hlt_fpr, hlt_tpr, _, _ = compute_roc(hlt_result['labels'], hlt_result['preds'], sample_weight=hlt_weight)
    teacher_fpr_30 = fpr_at_target_tpr(teacher_tpr, teacher_fpr, 0.30)
    teacher_fpr_50 = fpr_at_target_tpr(teacher_tpr, teacher_fpr, 0.50)
    hlt_fpr_30 = fpr_at_target_tpr(hlt_tpr, hlt_fpr, 0.30)
    hlt_fpr_50 = fpr_at_target_tpr(hlt_tpr, hlt_fpr, 0.50)

    for model_name in MODEL_DISPLAY_ORDER:
        model_result = repeat_result['models'][model_name]
        sample_weight = np.asarray(model_result['weights'], dtype=np.float64) if use_sample_weight_for_all_losses else None
        fpr, tpr, auc_raw, auc_weighted_raw = compute_roc(model_result['labels'], model_result['preds'], sample_weight=sample_weight)
        fpr_30 = fpr_at_target_tpr(tpr, fpr, 0.30)
        fpr_50 = fpr_at_target_tpr(tpr, fpr, 0.50)
        repeat_detail_rows.append({
            'repeat_seed': int(repeat_seed),
            'model': str(model_name),
            'auc': float(model_result.get('auc', auc_raw)),
            'auc_weighted': float(model_result.get('auc_weighted', auc_weighted_raw)),
            'fpr_at_tpr30': float(fpr_30),
            'fpr_at_tpr50': float(fpr_50),
            'gap_recovery_tpr30': float(gap_recovery(fpr_30, hlt_fpr_30, teacher_fpr_30)),
            'gap_recovery_tpr50': float(gap_recovery(fpr_50, hlt_fpr_50, teacher_fpr_50)),
            'ckpt_path': str(model_result.get('ckpt_path', '')),
            'prediction_path': str(model_result.get('prediction_path', '')),
            'epoch_metrics_path': str(model_result.get('epoch_metrics_path', '')),
        })

repeat_detail_df = pd.DataFrame(repeat_detail_rows)
summary_rows = []
for model_name in MODEL_DISPLAY_ORDER:
    df_model = repeat_detail_df[repeat_detail_df['model'] == str(model_name)].copy()
    if df_model.empty:
        continue
    summary_rows.append({
        'model': str(model_name),
        'auc_mean': float(df_model['auc'].mean()),
        'auc_std': float(df_model['auc'].std(ddof=0)),
        'auc_var': float(df_model['auc'].var(ddof=0)),
        'auc_weighted_mean': float(df_model['auc_weighted'].mean()),
        'auc_weighted_std': float(df_model['auc_weighted'].std(ddof=0)),
        'auc_weighted_var': float(df_model['auc_weighted'].var(ddof=0)),
        'fpr_at_tpr30_mean': float(df_model['fpr_at_tpr30'].mean()),
        'fpr_at_tpr30_std': float(df_model['fpr_at_tpr30'].std(ddof=0)),
        'fpr_at_tpr30_var': float(df_model['fpr_at_tpr30'].var(ddof=0)),
        'fpr_at_tpr50_mean': float(df_model['fpr_at_tpr50'].mean()),
        'fpr_at_tpr50_std': float(df_model['fpr_at_tpr50'].std(ddof=0)),
        'fpr_at_tpr50_var': float(df_model['fpr_at_tpr50'].var(ddof=0)),
        'gap_recovery_tpr30_mean': float(df_model['gap_recovery_tpr30'].mean()),
        'gap_recovery_tpr30_std': float(df_model['gap_recovery_tpr30'].std(ddof=0)),
        'gap_recovery_tpr30_var': float(df_model['gap_recovery_tpr30'].var(ddof=0)),
        'gap_recovery_tpr50_mean': float(df_model['gap_recovery_tpr50'].mean()),
        'gap_recovery_tpr50_std': float(df_model['gap_recovery_tpr50'].std(ddof=0)),
        'gap_recovery_tpr50_var': float(df_model['gap_recovery_tpr50'].var(ddof=0)),
    })
summary_df = pd.DataFrame(summary_rows)

repeat_detail_out = os.path.join(TABLE_DIR, 'joint_repeat_detail.csv')
repeat_summary_out = os.path.join(TABLE_DIR, 'joint_repeat_summary.csv')
save_rows_csv(repeat_detail_out, repeat_detail_rows)
save_rows_csv(repeat_summary_out, summary_rows)
print('Saved table:', repeat_detail_out)
print('Saved table:', repeat_summary_out)

common_tpr = np.linspace(0.0, 1.0, 401)
plt.figure(figsize=(7.4, 6.2))
for model_name in MODEL_DISPLAY_ORDER:
    fpr_rows = []
    auc_rows = []
    auc_weighted_rows = []
    for repeat_result in repeat_results:
        model_result = repeat_result['models'][model_name]
        sample_weight = np.asarray(model_result['weights'], dtype=np.float64) if use_sample_weight_for_all_losses else None
        fpr, tpr, auc_raw, auc_weighted_raw = compute_roc(model_result['labels'], model_result['preds'], sample_weight=sample_weight)
        fpr_rows.append(np.interp(common_tpr, tpr, fpr))
        auc_rows.append(float(model_result.get('auc', auc_raw)))
        auc_weighted_rows.append(float(model_result.get('auc_weighted', auc_weighted_raw)))
    fpr_arr = np.asarray(fpr_rows, dtype=np.float64)
    mean_fpr = fpr_arr.mean(axis=0)
    std_fpr = fpr_arr.std(axis=0)
    color = MODEL_COLORS.get(model_name, None)
    label = (
        f"{model_name} AUC={np.mean(auc_rows):.4f}+/-{np.std(auc_rows):.4f}, "
        f"wAUC={np.mean(auc_weighted_rows):.4f}+/-{np.std(auc_weighted_rows):.4f}"
    )
    plt.semilogy(common_tpr, np.clip(mean_fpr, 1e-6, 1.0), lw=2.0, color=color, label=label)
    plt.fill_between(
        common_tpr,
        np.clip(mean_fpr - std_fpr, 1e-6, 1.0),
        np.clip(mean_fpr + std_fpr, 1e-6, 1.0),
        color=color,
        alpha=0.12,
    )
plt.xlabel('True Positive Rate (Signal efficiency)')
plt.ylabel('False Positive Rate')
plt.title('Shared-encoder joint training ROC mean +/- std (test)')
plt.xlim(0.0, 1.0)
plt.ylim(1e-4, 1.0)
plt.grid(True, which='both', alpha=0.3)
plt.legend(loc='upper left', fontsize=8)
plt.tight_layout()
roc_out = os.path.join(FIG_DIR, 'sharedencoder_joint_downstream_roc_mean_std.png')
plt.savefig(roc_out, dpi=180, bbox_inches='tight')
print('Saved figure:', roc_out)
plt.show()

joint_only = repeat_detail_df[repeat_detail_df['model'].isin(['JointSharedEncoder(HLT,no_kd)', 'JointSharedEncoder(HLT,with_kd)'])].copy()
BEST_JOINT_ROW = joint_only.sort_values(['auc', 'auc_weighted'], ascending=False).iloc[0].to_dict()
print('Best joint model by test auc:', BEST_JOINT_ROW['model'], 'repeat_seed=', int(BEST_JOINT_ROW['repeat_seed']))
print('Best joint auc:', float(BEST_JOINT_ROW['auc']))
print('Best joint weighted auc:', float(BEST_JOINT_ROW['auc_weighted']))

summary_display_df = summary_df.copy()
for col in summary_display_df.columns:
    if col == 'model':
        continue
    if col.startswith('gap_recovery') or col.startswith('fpr_at_'):
        summary_display_df[col] = summary_display_df[col].map(lambda x: 'nan' if pd.isna(x) else f'{100.0 * float(x):.4f}%')
    else:
        summary_display_df[col] = summary_display_df[col].map(lambda x: f'{float(x):.6f}')

try:
    display(repeat_detail_df)
    display(summary_display_df)
except Exception:
    print(repeat_detail_df.to_string(index=False))
    print(summary_display_df.to_string(index=False))


In [ ]:
# Inspect feature reconstruction with the joint model that has the best test AUC.

best_joint_model_name = str(BEST_JOINT_ROW['model'])
best_joint_repeat_seed = int(BEST_JOINT_ROW['repeat_seed'])
best_joint_ckpt = str(BEST_JOINT_ROW['ckpt_path'])

best_joint_model = SharedEncoderUnsmearClassifier(**CONFIG['joint_model']).to(device)
load_checkpoint(best_joint_model, best_joint_ckpt, map_location=device)
best_joint_model.eval()

pred_best_joint_reco = predict_joint_reco(best_joint_model, test_loader_joint)
x_test_std = x_joint[test_idx]
y_test_std = y_joint[test_idx]
mask_test = common_mask[test_idx]

residual_sources = {
    'hlt': x_test_std - y_test_std,
    'best_joint': pred_best_joint_reco - y_test_std,
}
plot_labels = {
    'hlt': 'HLT baseline (post - pre)',
    'best_joint': f"{best_joint_model_name} | seed={best_joint_repeat_seed}",
}
plot_colors = {
    'hlt': '#4C78A8',
    'best_joint': '#54A24B' if 'with_kd' in best_joint_model_name else '#F58518',
}

metrics_rows = []
for feat_idx, feat_name in enumerate(feat_names):
    plt.figure(figsize=(6.6, 4.6))
    for method_name in ['hlt', 'best_joint']:
        residual = residual_sources[method_name][..., feat_idx][mask_test]
        residual = maybe_wrap_residual(feat_name, feat_idx, residual, scale=feat_stds[feat_idx])
        plt.hist(
            residual,
            bins=120,
            density=True,
            alpha=0.35,
            label=plot_labels[method_name],
            color=plot_colors[method_name],
        )
        mm_raw = metric_dict(residual)
        abs_residual = np.abs(np.asarray(residual, dtype=np.float64))
        mm = {
            'bias': float(mm_raw.get('bias', mm_raw.get('mean', np.mean(residual)))),
            'mae': float(mm_raw.get('mae', np.mean(abs_residual))),
            'rmse': float(mm_raw.get('rmse', np.sqrt(np.mean(np.square(residual))))),
            'abs_p50': float(mm_raw.get('abs_p50', mm_raw.get('p50_abs', np.percentile(abs_residual, 50)))),
            'abs_p90': float(mm_raw.get('abs_p90', mm_raw.get('p90_abs', np.percentile(abs_residual, 90)))),
            'abs_p99': float(mm_raw.get('abs_p99', np.percentile(abs_residual, 99))),
        }
        metrics_rows.append({
            'feature': feat_name,
            'method': method_name,
            'best_model': best_joint_model_name if method_name == 'best_joint' else 'hlt',
            'best_repeat_seed': best_joint_repeat_seed if method_name == 'best_joint' else np.nan,
            **mm,
        })

    plt.title(f'Residual compare: {feat_name}')
    plt.xlabel('Residual (std space)')
    plt.ylabel('Density')
    plt.grid(True, alpha=0.25)
    plt.legend()
    plt.tight_layout()
    out = os.path.join(FIG_DIR, f'joint_reco_residual_compare_best_{feat_name}.png')
    plt.savefig(out, dpi=160, bbox_inches='tight')
    print('Saved figure:', out)
    plt.show()

metrics_df = pd.DataFrame(metrics_rows)
metrics_df = metrics_df[['feature', 'method', 'best_model', 'best_repeat_seed', 'bias', 'mae', 'rmse', 'abs_p50', 'abs_p90', 'abs_p99']]

print()
print('=' * 120)
print(f'Metrics summary (std space) | split=test | best_joint={best_joint_model_name} | repeat_seed={best_joint_repeat_seed}')
print('=' * 120)
print(metrics_df.to_string(index=False, float_format=lambda x: f'{x:.6f}'))

metrics_out = os.path.join(TABLE_DIR, 'joint_reco_metrics_summary_best_auc_model_test.csv')
metrics_df.to_csv(metrics_out, index=False)
print('Saved table:', metrics_out)

try:
    display(metrics_df)
except Exception:
    pass


In [ ]:
# Gradient probe analysis from saved train/val probe tables (ported from joint_0301)
import json
import pandas as pd

TARGET_GRAD_REPEAT_SEED = int(CONFIG.get('repeat_seeds', [repeat_results[0]['repeat_seed']])[0])
if 'repeat_results' not in globals() or len(repeat_results) == 0:
    raise RuntimeError('repeat_results is empty. Run the training/repeat cell first.')
representative_repeat = None
for _repeat_result in repeat_results:
    if int(_repeat_result['repeat_seed']) == int(TARGET_GRAD_REPEAT_SEED):
        representative_repeat = _repeat_result
        break
if representative_repeat is None:
    representative_repeat = repeat_results[0]
    TARGET_GRAD_REPEAT_SEED = int(representative_repeat['repeat_seed'])

REPRESENTATIVE_GRAD_PROBE_DIR = os.path.join(str(representative_repeat['repeat_dir']), 'grad_probe')
GRAD_PROBE_PREFIXES = {
    'joint_no_kd': representative_repeat['models']['JointSharedEncoder(HLT,no_kd)'].get(
        'grad_probe_prefix',
        os.path.join(REPRESENTATIVE_GRAD_PROBE_DIR, 'joint_no_kd'),
    ),
    'joint_with_kd': representative_repeat['models']['JointSharedEncoder(HLT,with_kd)'].get(
        'grad_probe_prefix',
        os.path.join(REPRESENTATIVE_GRAD_PROBE_DIR, 'joint_with_kd'),
    ),
}
print('Representative gradient repeat seed:', TARGET_GRAD_REPEAT_SEED)
print('Representative grad probe dir:', REPRESENTATIVE_GRAD_PROBE_DIR)

group_order = ['shared_all', 'input_proj', 'layer_1', 'layer_last']
group_title = {
    'shared_all': 'shared_all',
    'input_proj': 'input_proj',
    'layer_1': 'layer_1',
    'layer_last': 'layer_last',
}
loss_order_all = ['unsmear', 'phys', 'hard', 'kd', 'attn']
loss_color = {
    'unsmear': '#4C78A8',
    'phys': '#9C755F',
    'hard': '#F58518',
    'kd': '#54A24B',
    'attn': '#E45756',
}
pair_order_all = [
    'unsmear_vs_phys',
    'unsmear_vs_hard',
    'unsmear_vs_kd',
    'unsmear_vs_attn',
    'phys_vs_hard',
    'phys_vs_kd',
    'phys_vs_attn',
    'hard_vs_kd',
    'hard_vs_attn',
    'kd_vs_attn',
]
pair_color = {
    'unsmear_vs_phys': '#72B7B2',
    'unsmear_vs_hard': '#E45756',
    'unsmear_vs_kd': '#EECA3B',
    'unsmear_vs_attn': '#B279A2',
    'phys_vs_hard': '#FF9DA6',
    'phys_vs_kd': '#9D755D',
    'phys_vs_attn': '#BAB0AC',
    'hard_vs_kd': '#54A24B',
    'hard_vs_attn': '#F58518',
    'kd_vs_attn': '#4C78A8',
}


def _grad_probe_paths(prefix: str):
    return {
        'scalar': f'{prefix}_scalar_losses.csv',
        'norm': f'{prefix}_grad_norms.csv',
        'cos': f'{prefix}_grad_cosines.csv',
        'feature_scalar': f'{prefix}_feature_scalar_losses.csv',
        'feature_norm': f'{prefix}_feature_grad_norms.csv',
        'feature_cos': f'{prefix}_feature_grad_cosines.csv',
        'meta': f'{prefix}_meta.json',
    }


def _load_grad_probe_tables(prefixes: dict[str, str]):
    scalar_frames = []
    norm_frames = []
    cos_frames = []
    feature_scalar_frames = []
    feature_norm_frames = []
    feature_cos_frames = []
    meta_map = {}
    missing = []
    for model_name, prefix in prefixes.items():
        paths = _grad_probe_paths(prefix)
        need = [paths['scalar'], paths['norm'], paths['cos']]
        missing_now = [p for p in need if not os.path.isfile(p)]
        if missing_now:
            missing.extend(missing_now)
            continue
        scalar_frames.append(pd.read_csv(paths['scalar']))
        norm_frames.append(pd.read_csv(paths['norm']))
        cos_frames.append(pd.read_csv(paths['cos']))
        feature_need = [paths['feature_scalar'], paths['feature_norm'], paths['feature_cos']]
        if all(os.path.isfile(p) for p in feature_need):
            feature_scalar_frames.append(pd.read_csv(paths['feature_scalar']))
            feature_norm_frames.append(pd.read_csv(paths['feature_norm']))
            feature_cos_frames.append(pd.read_csv(paths['feature_cos']))
        if os.path.isfile(paths['meta']):
            with open(paths['meta'], 'r', encoding='utf-8') as f:
                meta_map[model_name] = json.load(f)
        else:
            meta_map[model_name] = {}
    if missing:
        raise FileNotFoundError(
            'Gradient probe tables are missing. If the models were loaded from checkpoints without probe history, rerun training with loading disabled to regenerate them. Missing files: '\
            + ', '.join(sorted(set(missing)))
        )

    feature_scalar_columns = ['model', 'split', 'epoch', 'batch_idx', 'sample_idx', 'total_batches', 'batch_fraction', 'loss_component', 'scalar_loss', 'feature_weight']
    feature_norm_columns = ['model', 'split', 'epoch', 'batch_idx', 'sample_idx', 'total_batches', 'batch_fraction', 'group', 'group_module', 'loss_component', 'grad_norm', 'feature_weight']
    feature_cos_columns = ['model', 'split', 'epoch', 'batch_idx', 'sample_idx', 'total_batches', 'batch_fraction', 'group', 'group_module', 'pair', 'cosine']

    feature_scalar_df = pd.concat(feature_scalar_frames, ignore_index=True) if feature_scalar_frames else pd.DataFrame(columns=feature_scalar_columns)
    feature_norm_df = pd.concat(feature_norm_frames, ignore_index=True) if feature_norm_frames else pd.DataFrame(columns=feature_norm_columns)
    feature_cos_df = pd.concat(feature_cos_frames, ignore_index=True) if feature_cos_frames else pd.DataFrame(columns=feature_cos_columns)
    return (
        pd.concat(scalar_frames, ignore_index=True),
        pd.concat(norm_frames, ignore_index=True),
        pd.concat(cos_frames, ignore_index=True),
        feature_scalar_df,
        feature_norm_df,
        feature_cos_df,
        meta_map,
    )


grad_scalar_df, grad_norms_df, grad_cos_df, grad_feature_scalar_df, grad_feature_norms_df, grad_feature_cos_df, grad_probe_meta = _load_grad_probe_tables(GRAD_PROBE_PREFIXES)

for df in [grad_scalar_df, grad_norms_df, grad_cos_df, grad_feature_scalar_df, grad_feature_norms_df, grad_feature_cos_df]:
    if 'epoch' in df.columns:
        df['epoch'] = df['epoch'].astype(int)
    if 'batch_idx' in df.columns:
        df['batch_idx_1based'] = df['batch_idx'].astype(int) + 1

print('Gradient probe models:', sorted(grad_norms_df['model'].unique().tolist()))
print('Gradient probe splits:', sorted(grad_norms_df['split'].unique().tolist()))
print('Gradient probe epochs:', sorted(grad_norms_df['epoch'].unique().tolist()))
for model_name, meta in grad_probe_meta.items():
    print(f"Meta[{model_name}] =", meta)


MODEL_LOSS_ORDER = {
    'joint_no_kd': ['unsmear', 'phys', 'hard'],
    'joint_with_kd': ['unsmear', 'phys', 'hard', 'kd', 'attn'],
}
MODEL_PAIR_ORDER = {
    'joint_no_kd': ['unsmear_vs_phys', 'unsmear_vs_hard', 'phys_vs_hard'],
    'joint_with_kd': [
        'unsmear_vs_phys',
        'unsmear_vs_hard',
        'unsmear_vs_kd',
        'unsmear_vs_attn',
        'phys_vs_hard',
        'phys_vs_kd',
        'phys_vs_attn',
        'hard_vs_kd',
        'hard_vs_attn',
        'kd_vs_attn',
    ],
}

GRAD_LOSS_WEIGHTS = {
    'joint_no_kd': {
        'unsmear': float(CONFIG['training']['joint_unsmear_weight']),
        'phys': float(CONFIG['training']['joint_unsmear_weight']),
        'hard': float(CONFIG['training']['joint_cls_weight']),
    },
    'joint_with_kd': {
        'unsmear': float(CONFIG['training']['joint_unsmear_weight']),
        'phys': float(CONFIG['training']['joint_unsmear_weight']),
        'hard': float(CONFIG['training']['joint_cls_weight']) * float(1.0 - CONFIG['kd']['alpha_kd']),
        'kd': float(CONFIG['training']['joint_cls_weight']) * float(CONFIG['kd']['alpha_kd']),
        'attn': float(CONFIG['training']['joint_cls_weight']) * float(CONFIG['kd']['alpha_attn']),
    },
}


def _loss_weight(model_name: str, loss_name: str) -> float:
    return float(GRAD_LOSS_WEIGHTS.get(model_name, {}).get(loss_name, 1.0))


def _active_loss_order(model_name: str, df: pd.DataFrame):
    present = set(df['loss_component'].dropna().unique().tolist())
    return [name for name in MODEL_LOSS_ORDER.get(model_name, loss_order_all) if name in present]


def _active_pair_order(model_name: str, df: pd.DataFrame):
    present = set(df['pair'].dropna().unique().tolist())
    return [name for name in MODEL_PAIR_ORDER.get(model_name, pair_order_all) if name in present]


def _active_feature_loss_order(model_name: str, df: pd.DataFrame):
    present = set(df['loss_component'].dropna().unique().tolist())
    return [name for name in FEATURE_LOSS_ORDER.get(model_name, feat_names) if name in present]


def _active_feature_pair_order(model_name: str, df: pd.DataFrame):
    present = set(df['pair'].dropna().unique().tolist())
    return [name for name in FEATURE_PAIR_ORDER.get(model_name, []) if name in present]


grad_norms_df['loss_weight'] = [
    _loss_weight(model_name, loss_name)
    for model_name, loss_name in zip(grad_norms_df['model'], grad_norms_df['loss_component'])
]
grad_norms_df['weighted_grad_norm'] = grad_norms_df['grad_norm'] * grad_norms_df['loss_weight']
print('Applied loss weights:', GRAD_LOSS_WEIGHTS)

FEATURE_LOSS_ORDER = {model_name: list(meta.get('feature_names', feat_names)) for model_name, meta in grad_probe_meta.items()}
FEATURE_WEIGHT_MAP = {
    model_name: {
        feat_name: float(weight)
        for feat_name, weight in zip(
            meta.get('feature_names', feat_names),
            meta.get('feature_loss_weights', [1.0] * len(feat_names)),
        )
    }
    for model_name, meta in grad_probe_meta.items()
}
FEATURE_PAIR_ORDER = {
    model_name: [
        f'{feat_a}_vs_{feat_b}'
        for i, feat_a in enumerate(FEATURE_LOSS_ORDER.get(model_name, feat_names))
        for feat_b in FEATURE_LOSS_ORDER.get(model_name, feat_names)[i + 1:]
    ]
    for model_name in FEATURE_LOSS_ORDER
}
feature_palette = ['#4C78A8', '#F58518', '#54A24B', '#E45756', '#72B7B2', '#B279A2', '#FF9DA6']
feature_color = {
    feat_name: feature_palette[idx % len(feature_palette)]
    for idx, feat_name in enumerate(feat_names)
}
if not grad_feature_norms_df.empty:
    if 'feature_weight' not in grad_feature_norms_df.columns:
        grad_feature_norms_df['feature_weight'] = [
            FEATURE_WEIGHT_MAP.get(model_name, {}).get(loss_name, 1.0)
            for model_name, loss_name in zip(grad_feature_norms_df['model'], grad_feature_norms_df['loss_component'])
        ]
    grad_feature_norms_df['weighted_grad_norm'] = grad_feature_norms_df['grad_norm'] * grad_feature_norms_df['feature_weight']
else:
    grad_feature_norms_df['weighted_grad_norm'] = pd.Series(dtype=float)
print('Applied feature weights:', FEATURE_WEIGHT_MAP)
FEATURE_PROBE_MODELS = sorted(grad_feature_norms_df['model'].dropna().unique().tolist()) if 'model' in grad_feature_norms_df.columns else []
print('Feature probe models:', FEATURE_PROBE_MODELS)


def _save_show(fig, stem: str):
    out = os.path.join(FIG_DIR, stem)
    fig.savefig(out, dpi=170, bbox_inches='tight')
    print('Saved figure:', out)
    plt.show()


def plot_epoch_grad_norms(model_name: str, split: str, epoch: int):
    sub_all = grad_norms_df[
        (grad_norms_df['model'] == model_name)
        & (grad_norms_df['split'] == split)
        & (grad_norms_df['epoch'] == int(epoch))
    ].copy()
    if sub_all.empty:
        raise ValueError(f'No grad norm data found for model={model_name}, split={split}, epoch={epoch}')
    active_loss_order = _active_loss_order(model_name, sub_all)
    fig, axes = plt.subplots(2, 2, figsize=(13, 8.5), sharex=False)
    axes = axes.reshape(-1)
    for ax, group_name in zip(axes, group_order):
        group_df = sub_all[sub_all['group'] == group_name].copy()
        if group_df.empty:
            ax.axis('off')
            continue
        group_df = group_df.sort_values(['sample_idx', 'batch_idx'])
        for loss_name in active_loss_order:
            loss_df = group_df[group_df['loss_component'] == loss_name].copy()
            if loss_df.empty:
                continue
            ax.plot(
                loss_df['batch_idx_1based'].to_numpy(dtype=float),
                loss_df['weighted_grad_norm'].to_numpy(dtype=float),
                marker='o',
                lw=1.8,
                ms=4,
                color=loss_color.get(loss_name, '#888888'),
                label=f"{loss_name} x {loss_df['loss_weight'].iloc[0]:.3g}",
            )
        ax.set_title(f"{group_title[group_name]}\n{group_df['group_module'].iloc[0]}")
        ax.set_xlabel('Batch index in epoch')
        ax.set_ylabel('Weighted grad norm')
        ax.grid(True, alpha=0.25)
        ax.legend(fontsize=8)
    fig.suptitle(f'Weighted gradient norms by sampled batches | model={model_name} | split={split} | epoch={epoch}')
    fig.tight_layout()
    _save_show(fig, f'grad_norm_weighted_epoch_model-{model_name}_split-{split}_epoch-{int(epoch):03d}.png')


def plot_epoch_grad_cosines(model_name: str, split: str, epoch: int):
    sub_all = grad_cos_df[
        (grad_cos_df['model'] == model_name)
        & (grad_cos_df['split'] == split)
        & (grad_cos_df['epoch'] == int(epoch))
    ].copy()
    if sub_all.empty:
        raise ValueError(f'No grad cosine data found for model={model_name}, split={split}, epoch={epoch}')
    active_pair_order = _active_pair_order(model_name, sub_all)
    fig, axes = plt.subplots(2, 2, figsize=(13, 8.5), sharex=False)
    axes = axes.reshape(-1)
    for ax, group_name in zip(axes, group_order):
        group_df = sub_all[sub_all['group'] == group_name].copy()
        if group_df.empty:
            ax.axis('off')
            continue
        group_df = group_df.sort_values(['sample_idx', 'batch_idx'])
        for pair_name in active_pair_order:
            pair_df = group_df[group_df['pair'] == pair_name].copy()
            if pair_df.empty:
                continue
            ax.plot(
                pair_df['batch_idx_1based'].to_numpy(dtype=float),
                pair_df['cosine'].to_numpy(dtype=float),
                marker='o',
                lw=1.8,
                ms=4,
                color=pair_color.get(pair_name, '#888888'),
                label=pair_name,
            )
        ax.axhline(0.0, color='black', lw=1.0, alpha=0.6)
        ax.set_ylim(-1.05, 1.05)
        ax.set_title(f"{group_title[group_name]}\n{group_df['group_module'].iloc[0]}")
        ax.set_xlabel('Batch index in epoch')
        ax.set_ylabel('Cosine')
        ax.grid(True, alpha=0.25)
        ax.legend(fontsize=8)
    fig.suptitle(f'Gradient cosines by sampled batches | model={model_name} | split={split} | epoch={epoch}')
    fig.tight_layout()
    _save_show(fig, f'grad_cos_epoch_model-{model_name}_split-{split}_epoch-{int(epoch):03d}.png')


def plot_epoch_mean_grad_norms(model_name: str, split: str):
    sub_all = grad_norms_df[
        (grad_norms_df['model'] == model_name)
        & (grad_norms_df['split'] == split)
    ].copy()
    if sub_all.empty:
        raise ValueError(f'No grad norm data found for model={model_name}, split={split}')
    sub_all = sub_all.groupby(['epoch', 'group', 'group_module', 'loss_component'], as_index=False).agg(
        weighted_grad_norm=('weighted_grad_norm', 'mean'),
        loss_weight=('loss_weight', 'first'),
    )
    active_loss_order = _active_loss_order(model_name, sub_all)
    fig, axes = plt.subplots(2, 2, figsize=(13, 8.5), sharex=True)
    axes = axes.reshape(-1)
    for ax, group_name in zip(axes, group_order):
        group_df = sub_all[sub_all['group'] == group_name].copy()
        if group_df.empty:
            ax.axis('off')
            continue
        for loss_name in active_loss_order:
            loss_df = group_df[group_df['loss_component'] == loss_name].copy().sort_values('epoch')
            if loss_df.empty:
                continue
            ax.plot(
                loss_df['epoch'].to_numpy(dtype=int),
                loss_df['weighted_grad_norm'].to_numpy(dtype=float),
                marker='o',
                lw=2.0,
                ms=4,
                color=loss_color.get(loss_name, '#888888'),
                label=f"{loss_name} x {loss_df['loss_weight'].iloc[0]:.3g}",
            )
        ax.set_title(f"{group_title[group_name]}\n{group_df['group_module'].iloc[0]}")
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Mean weighted grad norm')
        ax.grid(True, alpha=0.25)
        ax.legend(fontsize=8)
    fig.suptitle(f'Mean weighted gradient norms across epochs | model={model_name} | split={split}')
    fig.tight_layout()
    _save_show(fig, f'grad_norm_weighted_epoch_mean_model-{model_name}_split-{split}.png')


def plot_epoch_mean_grad_cosines(model_name: str, split: str):
    sub_all = grad_cos_df[
        (grad_cos_df['model'] == model_name)
        & (grad_cos_df['split'] == split)
    ].copy()
    if sub_all.empty:
        raise ValueError(f'No grad cosine data found for model={model_name}, split={split}')
    sub_all = sub_all.groupby(['epoch', 'group', 'group_module', 'pair'], as_index=False).agg(
        cosine_mean=('cosine', 'mean'),
        cosine_std=('cosine', 'std'),
    )
    sub_all['cosine_std'] = sub_all['cosine_std'].fillna(0.0)
    active_pair_order = _active_pair_order(model_name, sub_all)
    fig, axes = plt.subplots(2, 2, figsize=(13, 8.5), sharex=True)
    axes = axes.reshape(-1)
    for ax, group_name in zip(axes, group_order):
        group_df = sub_all[sub_all['group'] == group_name].copy()
        if group_df.empty:
            ax.axis('off')
            continue
        for pair_name in active_pair_order:
            pair_df = group_df[group_df['pair'] == pair_name].copy().sort_values('epoch')
            if pair_df.empty:
                continue
            x = pair_df['epoch'].to_numpy(dtype=int)
            y = pair_df['cosine_mean'].to_numpy(dtype=float)
            y_std = pair_df['cosine_std'].to_numpy(dtype=float)
            color = pair_color.get(pair_name, '#888888')
            ax.plot(
                x,
                y,
                marker='o',
                lw=2.0,
                ms=4,
                color=color,
                label=f'{pair_name} mean',
            )
            ax.fill_between(
                x,
                np.clip(y - y_std, -1.0, 1.0),
                np.clip(y + y_std, -1.0, 1.0),
                color=color,
                alpha=0.18,
                label=f'{pair_name} std',
            )
        ax.axhline(0.0, color='black', lw=1.0, alpha=0.6)
        ax.set_ylim(-1.05, 1.05)
        ax.set_title(f"{group_title[group_name]}\n{group_df['group_module'].iloc[0]}")
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Cosine mean ± std')
        ax.grid(True, alpha=0.25)
        ax.legend(fontsize=8)
    fig.suptitle(f'Mean and std of gradient cosines across epochs | model={model_name} | split={split}')
    fig.tight_layout()
    _save_show(fig, f'grad_cos_epoch_mean_std_model-{model_name}_split-{split}.png')


def plot_feature_epoch_grad_norms(model_name: str, split: str, epoch: int):
    sub_all = grad_feature_norms_df[
        (grad_feature_norms_df['model'] == model_name)
        & (grad_feature_norms_df['split'] == split)
        & (grad_feature_norms_df['epoch'] == int(epoch))
    ].copy()
    if sub_all.empty:
        raise ValueError(f'No feature grad norm data found for model={model_name}, split={split}, epoch={epoch}')
    active_feature_order = _active_feature_loss_order(model_name, sub_all)
    fig, axes = plt.subplots(2, 2, figsize=(13, 8.5), sharex=False)
    axes = axes.reshape(-1)
    for ax, group_name in zip(axes, group_order):
        group_df = sub_all[sub_all['group'] == group_name].copy()
        if group_df.empty:
            ax.axis('off')
            continue
        group_df = group_df.sort_values(['sample_idx', 'batch_idx'])
        for feat_name in active_feature_order:
            feat_df = group_df[group_df['loss_component'] == feat_name].copy()
            if feat_df.empty:
                continue
            ax.plot(
                feat_df['batch_idx_1based'].to_numpy(dtype=float),
                feat_df['weighted_grad_norm'].to_numpy(dtype=float),
                marker='o',
                lw=1.8,
                ms=4,
                color=feature_color.get(feat_name, '#888888'),
                label=f"{feat_name} x {feat_df['feature_weight'].iloc[0]:.3g}",
            )
        ax.set_title(f"{group_title[group_name]}\n{group_df['group_module'].iloc[0]}")
        ax.set_xlabel('Batch index in epoch')
        ax.set_ylabel('Weighted grad norm')
        ax.grid(True, alpha=0.25)
        ax.legend(fontsize=8)
    fig.suptitle(f'Weighted feature gradient norms by sampled batches | model={model_name} | split={split} | epoch={epoch}')
    fig.tight_layout()
    _save_show(fig, f'feature_grad_norm_weighted_epoch_model-{model_name}_split-{split}_epoch-{int(epoch):03d}.png')


def plot_feature_epoch_grad_cosines(model_name: str, split: str, epoch: int):
    sub_all = grad_feature_cos_df[
        (grad_feature_cos_df['model'] == model_name)
        & (grad_feature_cos_df['split'] == split)
        & (grad_feature_cos_df['epoch'] == int(epoch))
    ].copy()
    if sub_all.empty:
        raise ValueError(f'No feature grad cosine data found for model={model_name}, split={split}, epoch={epoch}')
    active_pair_order = _active_feature_pair_order(model_name, sub_all)
    fig, axes = plt.subplots(2, 2, figsize=(13, 8.5), sharex=False)
    axes = axes.reshape(-1)
    for ax, group_name in zip(axes, group_order):
        group_df = sub_all[sub_all['group'] == group_name].copy()
        if group_df.empty:
            ax.axis('off')
            continue
        group_df = group_df.sort_values(['sample_idx', 'batch_idx'])
        for pair_name in active_pair_order:
            pair_df = group_df[group_df['pair'] == pair_name].copy()
            if pair_df.empty:
                continue
            ax.plot(
                pair_df['batch_idx_1based'].to_numpy(dtype=float),
                pair_df['cosine'].to_numpy(dtype=float),
                marker='o',
                lw=1.8,
                ms=4,
                color=feature_color.get(pair_name.split('_vs_')[0], '#888888'),
                label=pair_name,
            )
        ax.axhline(0.0, color='black', lw=1.0, alpha=0.6)
        ax.set_ylim(-1.05, 1.05)
        ax.set_title(f"{group_title[group_name]}\n{group_df['group_module'].iloc[0]}")
        ax.set_xlabel('Batch index in epoch')
        ax.set_ylabel('Cosine')
        ax.grid(True, alpha=0.25)
        ax.legend(fontsize=7)
    fig.suptitle(f'Feature gradient cosines by sampled batches | model={model_name} | split={split} | epoch={epoch}')
    fig.tight_layout()
    _save_show(fig, f'feature_grad_cos_epoch_model-{model_name}_split-{split}_epoch-{int(epoch):03d}.png')


def plot_feature_epoch_mean_grad_norms(model_name: str, split: str):
    sub_all = grad_feature_norms_df[
        (grad_feature_norms_df['model'] == model_name)
        & (grad_feature_norms_df['split'] == split)
    ].copy()
    if sub_all.empty:
        raise ValueError(f'No feature grad norm data found for model={model_name}, split={split}')
    sub_all = sub_all.groupby(['epoch', 'group', 'group_module', 'loss_component'], as_index=False).agg(
        weighted_grad_norm=('weighted_grad_norm', 'mean'),
        feature_weight=('feature_weight', 'first'),
    )
    active_feature_order = _active_feature_loss_order(model_name, sub_all)
    fig, axes = plt.subplots(2, 2, figsize=(13, 8.5), sharex=True)
    axes = axes.reshape(-1)
    for ax, group_name in zip(axes, group_order):
        group_df = sub_all[sub_all['group'] == group_name].copy()
        if group_df.empty:
            ax.axis('off')
            continue
        for feat_name in active_feature_order:
            feat_df = group_df[group_df['loss_component'] == feat_name].copy().sort_values('epoch')
            if feat_df.empty:
                continue
            ax.plot(
                feat_df['epoch'].to_numpy(dtype=int),
                feat_df['weighted_grad_norm'].to_numpy(dtype=float),
                marker='o',
                lw=2.0,
                ms=4,
                color=feature_color.get(feat_name, '#888888'),
                label=f"{feat_name} x {feat_df['feature_weight'].iloc[0]:.3g}",
            )
        ax.set_title(f"{group_title[group_name]}\n{group_df['group_module'].iloc[0]}")
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Mean weighted grad norm')
        ax.grid(True, alpha=0.25)
        ax.legend(fontsize=7)
    fig.suptitle(f'Mean weighted feature gradient norms across epochs | model={model_name} | split={split}')
    fig.tight_layout()
    _save_show(fig, f'feature_grad_norm_weighted_epoch_mean_model-{model_name}_split-{split}.png')


def plot_feature_epoch_mean_grad_cosines(model_name: str, split: str):
    sub_all = grad_feature_cos_df[
        (grad_feature_cos_df['model'] == model_name)
        & (grad_feature_cos_df['split'] == split)
    ].copy()
    if sub_all.empty:
        raise ValueError(f'No feature grad cosine data found for model={model_name}, split={split}')
    sub_all = sub_all.groupby(['epoch', 'group', 'group_module', 'pair'], as_index=False).agg(
        cosine_mean=('cosine', 'mean'),
        cosine_std=('cosine', 'std'),
    )
    sub_all['cosine_std'] = sub_all['cosine_std'].fillna(0.0)
    active_pair_order = _active_feature_pair_order(model_name, sub_all)
    fig, axes = plt.subplots(2, 2, figsize=(13, 8.5), sharex=True)
    axes = axes.reshape(-1)
    for ax, group_name in zip(axes, group_order):
        group_df = sub_all[sub_all['group'] == group_name].copy()
        if group_df.empty:
            ax.axis('off')
            continue
        for pair_name in active_pair_order:
            pair_df = group_df[group_df['pair'] == pair_name].copy().sort_values('epoch')
            if pair_df.empty:
                continue
            x = pair_df['epoch'].to_numpy(dtype=int)
            y = pair_df['cosine_mean'].to_numpy(dtype=float)
            y_std = pair_df['cosine_std'].to_numpy(dtype=float)
            color = feature_color.get(pair_name.split('_vs_')[0], '#888888')
            ax.plot(
                x,
                y,
                marker='o',
                lw=2.0,
                ms=4,
                color=color,
                label=f'{pair_name} mean',
            )
            ax.fill_between(
                x,
                np.clip(y - y_std, -1.0, 1.0),
                np.clip(y + y_std, -1.0, 1.0),
                color=color,
                alpha=0.18,
                label=f'{pair_name} std',
            )
        ax.axhline(0.0, color='black', lw=1.0, alpha=0.6)
        ax.set_ylim(-1.05, 1.05)
        ax.set_title(f"{group_title[group_name]}\n{group_df['group_module'].iloc[0]}")
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Cosine mean ± std')
        ax.grid(True, alpha=0.25)
        ax.legend(fontsize=7)
    fig.suptitle(f'Mean and std of feature gradient cosines across epochs | model={model_name} | split={split}')
    fig.tight_layout()
    _save_show(fig, f'feature_grad_cos_epoch_mean_std_model-{model_name}_split-{split}.png')


GRAD_PLOT_MODEL = 'joint_with_kd'
_available_epochs = sorted(grad_norms_df[grad_norms_df['model'] == GRAD_PLOT_MODEL]['epoch'].unique().tolist())
if not _available_epochs:
    raise ValueError(f'No gradient probe epochs found for model={GRAD_PLOT_MODEL}')
GRAD_PLOT_EPOCH = int(_available_epochs[min(20, len(_available_epochs) - 1)])
print('Gradient plot model:', GRAD_PLOT_MODEL)
print('Gradient plot epoch:', GRAD_PLOT_EPOCH)

for split_name in ['train', 'val']:
    plot_epoch_grad_norms(GRAD_PLOT_MODEL, split_name, GRAD_PLOT_EPOCH)
    plot_epoch_grad_cosines(GRAD_PLOT_MODEL, split_name, GRAD_PLOT_EPOCH)
    plot_epoch_mean_grad_norms(GRAD_PLOT_MODEL, split_name)
    plot_epoch_mean_grad_cosines(GRAD_PLOT_MODEL, split_name)
    if GRAD_PLOT_MODEL in FEATURE_PROBE_MODELS:
        plot_feature_epoch_grad_norms(GRAD_PLOT_MODEL, split_name, GRAD_PLOT_EPOCH)
        plot_feature_epoch_grad_cosines(GRAD_PLOT_MODEL, split_name, GRAD_PLOT_EPOCH)
        plot_feature_epoch_mean_grad_norms(GRAD_PLOT_MODEL, split_name)
        plot_feature_epoch_mean_grad_cosines(GRAD_PLOT_MODEL, split_name)
    else:
        print(f'Skip feature-level plots for model={GRAD_PLOT_MODEL}: no feature probe tables found.')


try:
    from IPython.display import display
    display(grad_scalar_df.head())
    display(grad_norms_df.head())
    display(grad_cos_df.head())
    display(grad_feature_scalar_df.head())
    display(grad_feature_norms_df.head())
    display(grad_feature_cos_df.head())
except Exception:
    pass





In [ ]:
# The no-fusion notebook does not run fusion-ratio diagnostics; that analysis is in joint_fusion/unsmear.ipynb.

In [ ]:
# Compare teacher embedding distances from the offline target for HLT inputs and both joint reconstructions.
# This cell only depends on earlier data loading, config, and dataset cells, not the training summary cell.

import os


TARGET_REPEAT_SEED = 42
TARGET_SPLIT = 'test'  # options: 'train' / 'val' / 'test'
TARGET_BATCH_SIZE = int(CONFIG['training'].get('batch_size', 256))
TARGET_RUN_NAME = str(globals().get('RUN_NAME', 'unsmear_transformer_sharedencoder_no_fusion_repeat3'))

if TARGET_SPLIT not in {'train', 'val', 'test'}:
    raise ValueError(f'Unsupported TARGET_SPLIT: {TARGET_SPLIT}')

run_dir = Path(REPEAT_DIR)
if run_dir.name != 'repeats':
    run_dir = Path(MODULE_DIR) / 'runs' / TARGET_RUN_NAME / 'repeats'
if not run_dir.is_dir():
    raise FileNotFoundError(f'Repeat directory not found: {run_dir}')

repeat_candidates = sorted([p for p in run_dir.iterdir() if p.is_dir() and p.name.endswith(f'_seed_{int(TARGET_REPEAT_SEED)}')])
if not repeat_candidates:
    raise FileNotFoundError(f'No repeat folder found for seed={int(TARGET_REPEAT_SEED)} under {run_dir}')
if len(repeat_candidates) > 1:
    print('Multiple repeat folders matched this seed; use the latest one:', repeat_candidates[-1])
repeat_dir = repeat_candidates[-1]
repeat_ckpt_dir = repeat_dir / 'ckpts'

teacher_ckpt = repeat_ckpt_dir / 'teacher_offline.pt'
joint_no_kd_ckpt = repeat_ckpt_dir / 'joint_sharedencoder_no_kd.pt'
joint_with_kd_ckpt = repeat_ckpt_dir / 'joint_sharedencoder_with_kd.pt'
for required_path in [teacher_ckpt, joint_no_kd_ckpt, joint_with_kd_ckpt]:
    if not required_path.is_file():
        raise FileNotFoundError(f'Checkpoint not found: {required_path}')

split_to_idx = {
    'train': train_idx,
    'val': val_idx,
    'test': test_idx,
}
sel_idx = split_to_idx[TARGET_SPLIT]
analysis_ds = JointJetDataset(
    x_joint[sel_idx],
    y_joint[sel_idx],
    common_mask[sel_idx],
    labels[sel_idx],
    weights[sel_idx],
)
analysis_loader = DataLoader(analysis_ds, batch_size=TARGET_BATCH_SIZE, shuffle=False)

teacher_model = ParticleTransformerKD(**CONFIG['tagger']).to(device)
load_checkpoint(teacher_model, str(teacher_ckpt), map_location=device)
teacher_model.eval()

joint_no_kd_model = SharedEncoderUnsmearClassifier(**CONFIG['joint_model']).to(device)
load_checkpoint(joint_no_kd_model, str(joint_no_kd_ckpt), map_location=device)
joint_no_kd_model.eval()

joint_with_kd_model = SharedEncoderUnsmearClassifier(**CONFIG['joint_model']).to(device)
load_checkpoint(joint_with_kd_model, str(joint_with_kd_ckpt), map_location=device)
joint_with_kd_model.eval()

distance_df = collect_embedding_distance_rows(
    teacher_model,
    joint_no_kd_model,
    joint_with_kd_model,
    analysis_loader,
)
if distance_df.empty:
    raise RuntimeError('No teacher embedding distance rows were collected.')

summary_rows = []
for method_name, df_method in distance_df.groupby('method', sort=False):
    values = df_method['teacher_embedding_distance'].to_numpy(dtype=np.float64)
    weights_np = df_method['weight'].to_numpy(dtype=np.float64)
    weighted_mean = float(np.average(values, weights=weights_np)) if np.sum(weights_np) > 0 else float(np.mean(values))
    summary_rows.append({
        'run_name': str(TARGET_RUN_NAME),
        'repeat_seed': int(TARGET_REPEAT_SEED),
        'split': str(TARGET_SPLIT),
        'method': str(method_name),
        'n_samples': int(len(df_method)),
        'distance_mean': float(np.mean(values)),
        'distance_std': float(np.std(values, ddof=0)),
        'distance_weighted_mean': weighted_mean,
        'distance_p50': float(np.quantile(values, 0.50)),
        'distance_p90': float(np.quantile(values, 0.90)),
        'distance_p99': float(np.quantile(values, 0.99)),
        'distance_max': float(np.max(values)),
    })
summary_df = pd.DataFrame(summary_rows)

method_order = ['hlt_input', 'joint_no_kd_reco', 'joint_with_kd_reco']
summary_df['method'] = pd.Categorical(summary_df['method'], categories=method_order, ordered=True)
summary_df = summary_df.sort_values('method').reset_index(drop=True)

display_name_map = {
    'hlt_input': 'HLT input',
    'joint_no_kd_reco': 'Joint no_kd reco',
    'joint_with_kd_reco': 'Joint with_kd reco',
}
plot_color_map = {
    'hlt_input': '#4C78A8',
    'joint_no_kd_reco': '#F58518',
    'joint_with_kd_reco': '#54A24B',
}

pivot_df = distance_df.pivot(index='sample_index', columns='method', values='teacher_embedding_distance').reset_index()
required_methods = ['hlt_input', 'joint_no_kd_reco', 'joint_with_kd_reco']
missing_methods = [m for m in required_methods if m not in pivot_df.columns]
if missing_methods:
    raise RuntimeError(f'Missing methods in pivot table: {missing_methods}')

improve_rows = []
for method_name in ['joint_no_kd_reco', 'joint_with_kd_reco']:
    improvement = pivot_df['hlt_input'].to_numpy(dtype=np.float64) - pivot_df[method_name].to_numpy(dtype=np.float64)
    improve_rows.append({
        'method': method_name,
        'delta_mean_vs_hlt': float(np.mean(improvement)),
        'delta_p50_vs_hlt': float(np.quantile(improvement, 0.50)),
        'delta_p90_vs_hlt': float(np.quantile(improvement, 0.90)),
        'fraction_better_than_hlt': float(np.mean(improvement > 0.0)),
        'fraction_worse_than_hlt': float(np.mean(improvement < 0.0)),
    })
improve_df = pd.DataFrame(improve_rows)

summary_view = summary_df.copy()
summary_view['method'] = summary_view['method'].map(display_name_map)
summary_view = summary_view[['method', 'distance_mean', 'distance_weighted_mean', 'distance_p50', 'distance_p90', 'distance_p99', 'distance_max']]
summary_view = summary_view.sort_values('distance_mean', ascending=True).reset_index(drop=True)

improve_view = improve_df.copy()
improve_view['method'] = improve_view['method'].map(display_name_map)
improve_view = improve_view.sort_values('delta_mean_vs_hlt', ascending=False).reset_index(drop=True)

safe_split = str(TARGET_SPLIT)
rows_out = os.path.join(TABLE_DIR, f'teacher_embedding_distance_rows_seed_{int(TARGET_REPEAT_SEED)}_{safe_split}.csv')
summary_out = os.path.join(TABLE_DIR, f'teacher_embedding_distance_summary_seed_{int(TARGET_REPEAT_SEED)}_{safe_split}.csv')
improve_out = os.path.join(TABLE_DIR, f'teacher_embedding_distance_improvement_seed_{int(TARGET_REPEAT_SEED)}_{safe_split}.csv')
distance_df.to_csv(rows_out, index=False)
summary_df.to_csv(summary_out, index=False)
improve_df.to_csv(improve_out, index=False)

best_method = str(summary_view.iloc[0]['method'])
print('Run name:', TARGET_RUN_NAME)
print('Repeat dir:', repeat_dir)
print('Teacher checkpoint:', teacher_ckpt)
print('Joint no_kd checkpoint:', joint_no_kd_ckpt)
print('Joint with_kd checkpoint:', joint_with_kd_ckpt)
print('Split:', TARGET_SPLIT)
print()
print('Closer to offline teacher embedding means smaller distance.')
print('Best method by mean distance:', best_method)
print()
print('Distance summary:')
print(summary_view.to_string(index=False, float_format=lambda x: f'{x:.6f}'))
print()
print('Improvement over HLT (positive means better than HLT):')
print(improve_view.to_string(index=False, float_format=lambda x: f'{x:.6f}'))
print()
print('Saved rows:', rows_out)
print('Saved summary:', summary_out)
print('Saved improvement:', improve_out)

display(summary_view)
display(improve_view)
display(distance_df.head())

fig, axes = plt.subplots(1, 2, figsize=(13.2, 4.8))

bar_methods = summary_view['method'].tolist()
bar_means = summary_view['distance_mean'].to_numpy(dtype=float)
bar_medians = summary_view['distance_p50'].to_numpy(dtype=float)
bar_colors = [plot_color_map[k] for k in ['hlt_input', 'joint_no_kd_reco', 'joint_with_kd_reco'] if display_name_map[k] in bar_methods]
axes[0].bar(bar_methods, bar_means, color=bar_colors, alpha=0.82, label='mean')
axes[0].scatter(bar_methods, bar_medians, color='black', s=42, zorder=3, label='median (p50)')
axes[0].set_title(f'Teacher embedding distance summary | seed={int(TARGET_REPEAT_SEED)} | split={TARGET_SPLIT}')
axes[0].set_ylabel('Distance to offline target')
axes[0].grid(True, axis='y', alpha=0.25)
axes[0].legend()
axes[0].tick_params(axis='x', rotation=10)

for method_name in ['joint_no_kd_reco', 'joint_with_kd_reco']:
    improvement = pivot_df['hlt_input'].to_numpy(dtype=np.float64) - pivot_df[method_name].to_numpy(dtype=np.float64)
    axes[1].hist(
        improvement,
        bins=80,
        alpha=0.40,
        density=True,
        label=f"{display_name_map[method_name]} - HLT improvement",
        color=plot_color_map[method_name],
    )
axes[1].axvline(0.0, color='black', ls='--', lw=1.5)
axes[1].set_title('Per-sample improvement over HLT')
axes[1].set_xlabel('HLT distance - model distance')
axes[1].set_ylabel('Density')
axes[1].grid(True, alpha=0.25)
axes[1].legend()

fig.tight_layout()
fig_out = os.path.join(FIG_DIR, f'teacher_embedding_distance_compare_seed_{int(TARGET_REPEAT_SEED)}_{safe_split}.png')
plt.savefig(fig_out, dpi=160, bbox_inches='tight')
print('Saved figure:', fig_out)
plt.show()

In [ ]:
# Analyze misclassified cases for the HLT baseline and joint_no_kd when the offline teacher is correct,
# and compare their teacher embedding distances.
# This cell only depends on earlier data loading, config, and dataset cells, not the training summary cell.

import os


TARGET_REPEAT_SEED = 42
TARGET_SPLIT = 'test'  # options: 'train' / 'val' / 'test'
TARGET_BATCH_SIZE = int(CONFIG['training'].get('batch_size', 256))
TARGET_RUN_NAME = str(globals().get('RUN_NAME', 'unsmear_transformer_sharedencoder_no_fusion_repeat3'))

if TARGET_SPLIT not in {'train', 'val', 'test'}:
    raise ValueError(f'Unsupported TARGET_SPLIT: {TARGET_SPLIT}')

run_dir = Path(REPEAT_DIR)
if run_dir.name != 'repeats':
    run_dir = Path(MODULE_DIR) / 'runs' / TARGET_RUN_NAME / 'repeats'
if not run_dir.is_dir():
    raise FileNotFoundError(f'Repeat directory not found: {run_dir}')

repeat_candidates = sorted([p for p in run_dir.iterdir() if p.is_dir() and p.name.endswith(f'_seed_{int(TARGET_REPEAT_SEED)}')])
if not repeat_candidates:
    raise FileNotFoundError(f'No repeat folder found for seed={int(TARGET_REPEAT_SEED)} under {run_dir}')
repeat_dir = repeat_candidates[-1]
repeat_ckpt_dir = repeat_dir / 'ckpts'

teacher_ckpt = repeat_ckpt_dir / 'teacher_offline.pt'
hlt_ckpt = repeat_ckpt_dir / 'student_hlt.pt'
joint_no_kd_ckpt = repeat_ckpt_dir / 'joint_sharedencoder_no_kd.pt'
for required_path in [teacher_ckpt, hlt_ckpt, joint_no_kd_ckpt]:
    if not required_path.is_file():
        raise FileNotFoundError(f'Checkpoint not found: {required_path}')

split_to_idx = {
    'train': train_idx,
    'val': val_idx,
    'test': test_idx,
}
sel_idx = split_to_idx[TARGET_SPLIT]
analysis_ds = JointJetDataset(
    x_joint[sel_idx],
    y_joint[sel_idx],
    common_mask[sel_idx],
    labels[sel_idx],
    weights[sel_idx],
)
analysis_loader = DataLoader(analysis_ds, batch_size=TARGET_BATCH_SIZE, shuffle=False)

teacher_model = ParticleTransformerKD(**CONFIG['tagger']).to(device)
load_checkpoint(teacher_model, str(teacher_ckpt), map_location=device)
teacher_model.eval()

hlt_model = ParticleTransformerKD(**CONFIG['tagger']).to(device)
load_checkpoint(hlt_model, str(hlt_ckpt), map_location=device)
hlt_model.eval()

joint_no_kd_model = SharedEncoderUnsmearClassifier(**CONFIG['joint_model']).to(device)
load_checkpoint(joint_no_kd_model, str(joint_no_kd_ckpt), map_location=device)
joint_no_kd_model.eval()

case_df = collect_case_rows(teacher_model, hlt_model, joint_no_kd_model, analysis_loader)
if case_df.empty:
    raise RuntimeError('No teacher-correct cases were collected.')

focus_order = ['hlt_wrong_joint_correct', 'hlt_correct_joint_wrong', 'both_wrong']
focus_df = case_df[case_df['case_group'].isin(focus_order)].copy()
if focus_df.empty:
    raise RuntimeError('No teacher-correct misclassified cases were found for the selected split.')

summary_rows = []
for case_group in focus_order:
    df_group = focus_df[focus_df['case_group'] == case_group].copy()
    if df_group.empty:
        continue
    hlt_values = df_group['hlt_teacher_embedding_distance'].to_numpy(dtype=np.float64)
    joint_values = df_group['joint_no_kd_teacher_embedding_distance'].to_numpy(dtype=np.float64)
    improve_values = df_group['distance_improvement_vs_hlt'].to_numpy(dtype=np.float64)
    weights_np = df_group['weight'].to_numpy(dtype=np.float64)
    summary_rows.append({
        'case_group': str(case_group),
        'n_samples': int(len(df_group)),
        'hlt_mean_distance': float(np.mean(hlt_values)),
        'joint_mean_distance': float(np.mean(joint_values)),
        'hlt_weighted_mean_distance': float(np.average(hlt_values, weights=weights_np)) if np.sum(weights_np) > 0 else float(np.mean(hlt_values)),
        'joint_weighted_mean_distance': float(np.average(joint_values, weights=weights_np)) if np.sum(weights_np) > 0 else float(np.mean(joint_values)),
        'hlt_p50_distance': float(np.quantile(hlt_values, 0.50)),
        'joint_p50_distance': float(np.quantile(joint_values, 0.50)),
        'improvement_mean': float(np.mean(improve_values)),
        'improvement_p50': float(np.quantile(improve_values, 0.50)),
        'fraction_joint_closer_than_hlt': float(np.mean(improve_values > 0.0)),
    })
summary_df = pd.DataFrame(summary_rows)

group_display_map = {
    'hlt_wrong_joint_correct': 'HLT wrong / Joint correct',
    'hlt_correct_joint_wrong': 'HLT correct / Joint wrong',
    'both_wrong': 'Both wrong',
    'both_correct': 'Both correct',
}
summary_view = summary_df.copy()
summary_view['case_group'] = summary_view['case_group'].map(group_display_map)

safe_split = str(TARGET_SPLIT)
rows_out = os.path.join(TABLE_DIR, f'teacher_embedding_case_rows_seed_{int(TARGET_REPEAT_SEED)}_{safe_split}.csv')
summary_out = os.path.join(TABLE_DIR, f'teacher_embedding_case_summary_seed_{int(TARGET_REPEAT_SEED)}_{safe_split}.csv')
focus_df.to_csv(rows_out, index=False)
summary_df.to_csv(summary_out, index=False)

print('Run name:', TARGET_RUN_NAME)
print('Repeat dir:', repeat_dir)
print('Teacher checkpoint:', teacher_ckpt)
print('HLT checkpoint:', hlt_ckpt)
print('Joint no_kd checkpoint:', joint_no_kd_ckpt)
print('Split:', TARGET_SPLIT)
print()
print('All case counts within teacher-correct samples:')
print(case_df['case_group'].value_counts().rename(index=group_display_map).to_string())
print()
print('Focused misclassified case summary:')
print(summary_view.to_string(index=False, float_format=lambda x: f'{x:.6f}'))
print()
print('Interpretation: smaller distance means closer to offline teacher embedding; positive improvement means joint_no_kd is closer than HLT.')
print('Saved rows:', rows_out)
print('Saved summary:', summary_out)

display(summary_view)
display(focus_df.head())

plot_groups = [g for g in focus_order if g in set(focus_df['case_group'])]
fig, axes = plt.subplots(1, 2, figsize=(13.8, 4.8))

x_pos = np.arange(len(plot_groups), dtype=float)
width = 0.36
hlt_means = [float(summary_df.loc[summary_df['case_group'] == g, 'hlt_mean_distance'].iloc[0]) for g in plot_groups]
joint_means = [float(summary_df.loc[summary_df['case_group'] == g, 'joint_mean_distance'].iloc[0]) for g in plot_groups]
axes[0].bar(x_pos - width / 2, hlt_means, width=width, color='#4C78A8', alpha=0.82, label='HLT input')
axes[0].bar(x_pos + width / 2, joint_means, width=width, color='#F58518', alpha=0.82, label='Joint no_kd reco')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels([group_display_map[g] for g in plot_groups], rotation=10)
axes[0].set_ylabel('Teacher embedding distance')
axes[0].set_title('Mean distance by misclassification case')
axes[0].grid(True, axis='y', alpha=0.25)
axes[0].legend()

for case_group in plot_groups:
    improvement = focus_df.loc[focus_df['case_group'] == case_group, 'distance_improvement_vs_hlt'].to_numpy(dtype=float)
    axes[1].hist(
        improvement,
        bins=70,
        alpha=0.42,
        density=True,
        label=group_display_map[case_group],
    )
axes[1].axvline(0.0, color='black', ls='--', lw=1.5)
axes[1].set_xlabel('HLT distance - Joint no_kd distance')
axes[1].set_ylabel('Density')
axes[1].set_title('Per-sample improvement within misclassified cases')
axes[1].grid(True, alpha=0.25)
axes[1].legend(fontsize=9)

fig.suptitle(f'Teacher-correct error-case comparison | seed={int(TARGET_REPEAT_SEED)} | split={TARGET_SPLIT}')
fig.tight_layout()
fig_out = os.path.join(FIG_DIR, f'teacher_embedding_error_case_compare_seed_{int(TARGET_REPEAT_SEED)}_{safe_split}.png')
plt.savefig(fig_out, dpi=160, bbox_inches='tight')
print('Saved figure:', fig_out)
plt.show()

In [ ]:
# Analyze teacher embedding distances for hlt / joint_no_kd / joint_with_kd
# and whether they correlate with each method logit gap from the offline teacher logit.
# This cell only depends on earlier data loading, config, and dataset cells, not the training summary cell.

import os


TARGET_REPEAT_SEED = 42
TARGET_SPLIT = 'test'  # options: 'train' / 'val' / 'test'
TARGET_BATCH_SIZE = int(CONFIG['training'].get('batch_size', 256))
TARGET_RUN_NAME = str(globals().get('RUN_NAME', 'unsmear_transformer_sharedencoder_no_fusion_repeat3'))

if TARGET_SPLIT not in {'train', 'val', 'test'}:
    raise ValueError(f'Unsupported TARGET_SPLIT: {TARGET_SPLIT}')

run_dir = Path(REPEAT_DIR)
if run_dir.name != 'repeats':
    run_dir = Path(MODULE_DIR) / 'runs' / TARGET_RUN_NAME / 'repeats'
if not run_dir.is_dir():
    raise FileNotFoundError(f'Repeat directory not found: {run_dir}')

repeat_candidates = sorted([p for p in run_dir.iterdir() if p.is_dir() and p.name.endswith(f'_seed_{int(TARGET_REPEAT_SEED)}')])
if not repeat_candidates:
    raise FileNotFoundError(f'No repeat folder found for seed={int(TARGET_REPEAT_SEED)} under {run_dir}')
repeat_dir = repeat_candidates[-1]
repeat_ckpt_dir = repeat_dir / 'ckpts'

teacher_ckpt = repeat_ckpt_dir / 'teacher_offline.pt'
hlt_ckpt = repeat_ckpt_dir / 'student_hlt.pt'
joint_no_kd_ckpt = repeat_ckpt_dir / 'joint_sharedencoder_no_kd.pt'
joint_with_kd_ckpt = repeat_ckpt_dir / 'joint_sharedencoder_with_kd.pt'
for required_path in [teacher_ckpt, hlt_ckpt, joint_no_kd_ckpt, joint_with_kd_ckpt]:
    if not required_path.is_file():
        raise FileNotFoundError(f'Checkpoint not found: {required_path}')

split_to_idx = {
    'train': train_idx,
    'val': val_idx,
    'test': test_idx,
}
sel_idx = split_to_idx[TARGET_SPLIT]
analysis_ds = JointJetDataset(
    x_joint[sel_idx],
    y_joint[sel_idx],
    common_mask[sel_idx],
    labels[sel_idx],
    weights[sel_idx],
)
analysis_loader = DataLoader(analysis_ds, batch_size=TARGET_BATCH_SIZE, shuffle=False)

teacher_model = ParticleTransformerKD(**CONFIG['tagger']).to(device)
load_checkpoint(teacher_model, str(teacher_ckpt), map_location=device)
teacher_model.eval()

hlt_model = ParticleTransformerKD(**CONFIG['tagger']).to(device)
load_checkpoint(hlt_model, str(hlt_ckpt), map_location=device)
hlt_model.eval()

joint_no_kd_model = SharedEncoderUnsmearClassifier(**CONFIG['joint_model']).to(device)
load_checkpoint(joint_no_kd_model, str(joint_no_kd_ckpt), map_location=device)
joint_no_kd_model.eval()

joint_with_kd_model = SharedEncoderUnsmearClassifier(**CONFIG['joint_model']).to(device)
load_checkpoint(joint_with_kd_model, str(joint_with_kd_ckpt), map_location=device)
joint_with_kd_model.eval()

distance_logit_df = collect_distance_logit_rows(
    teacher_model,
    hlt_model,
    joint_no_kd_model,
    joint_with_kd_model,
    analysis_loader,
)
if distance_logit_df.empty:
    raise RuntimeError('No distance/logit rows were collected.')

method_order = ['hlt_input', 'joint_no_kd_reco', 'joint_with_kd_reco']
display_name_map = {
    'hlt_input': 'HLT input',
    'joint_no_kd_reco': 'Joint no_kd reco',
    'joint_with_kd_reco': 'Joint with_kd reco',
}
plot_color_map = {
    'hlt_input': '#4C78A8',
    'joint_no_kd_reco': '#F58518',
    'joint_with_kd_reco': '#54A24B',
}

summary_rows = []
binned_curves = {}
for method_name in method_order:
    df_method = distance_logit_df[distance_logit_df['method'] == method_name].copy()
    if df_method.empty:
        continue
    summary_rows.append({
        'method': str(method_name),
        'n_samples': int(len(df_method)),
        'distance_mean': float(df_method['teacher_embedding_distance'].mean()),
        'abs_logit_gap_mean': float(df_method['logit_gap_abs'].mean()),
        'abs_logit_gap_weighted_mean': float(np.average(df_method['logit_gap_abs'].to_numpy(dtype=np.float64), weights=df_method['weight'].to_numpy(dtype=np.float64))) if float(df_method['weight'].sum()) > 0 else float(df_method['logit_gap_abs'].mean()),
        'abs_prob_gap_mean': float(df_method['prob_gap_abs'].mean()),
        'abs_prob_gap_weighted_mean': float(np.average(df_method['prob_gap_abs'].to_numpy(dtype=np.float64), weights=df_method['weight'].to_numpy(dtype=np.float64))) if float(df_method['weight'].sum()) > 0 else float(df_method['prob_gap_abs'].mean()),
        'pearson_distance_vs_abs_gap': corr_safe(df_method['teacher_embedding_distance'], df_method['logit_gap_abs'], method='pearson'),
        'spearman_distance_vs_abs_gap': corr_safe(df_method['teacher_embedding_distance'], df_method['logit_gap_abs'], method='spearman'),
        'pearson_distance_vs_signed_gap': corr_safe(df_method['teacher_embedding_distance'], df_method['logit_gap_signed'], method='pearson'),
        'spearman_distance_vs_signed_gap': corr_safe(df_method['teacher_embedding_distance'], df_method['logit_gap_signed'], method='spearman'),
        'pearson_distance_vs_abs_prob_gap': corr_safe(df_method['teacher_embedding_distance'], df_method['prob_gap_abs'], method='pearson'),
        'spearman_distance_vs_abs_prob_gap': corr_safe(df_method['teacher_embedding_distance'], df_method['prob_gap_abs'], method='spearman'),
    })
    binned_curves[method_name] = build_binned_curve(df_method, n_bins=12)
summary_df = pd.DataFrame(summary_rows)
summary_df['method'] = pd.Categorical(summary_df['method'], categories=method_order, ordered=True)
summary_df = summary_df.sort_values('method').reset_index(drop=True)
summary_view = summary_df.copy()
summary_view['method'] = summary_view['method'].map(display_name_map)

safe_split = str(TARGET_SPLIT)
rows_out = os.path.join(TABLE_DIR, f'teacher_embedding_logit_relation_rows_seed_{int(TARGET_REPEAT_SEED)}_{safe_split}.csv')
summary_out = os.path.join(TABLE_DIR, f'teacher_embedding_logit_relation_summary_seed_{int(TARGET_REPEAT_SEED)}_{safe_split}.csv')
distance_logit_df.to_csv(rows_out, index=False)
summary_df.to_csv(summary_out, index=False)

print('Run name:', TARGET_RUN_NAME)
print('Repeat dir:', repeat_dir)
print('Split:', TARGET_SPLIT)
print()
print('Question being tested: does larger teacher embedding distance correspond to larger logit gap from offline teacher?')
print('For logit gap, smaller is better; for distance, smaller is better.')
print()
print(summary_view.to_string(index=False, float_format=lambda x: f'{x:.6f}'))
print()
print('Saved rows:', rows_out)
print('Saved summary:', summary_out)

display(summary_view)
display(distance_logit_df.head())

fig, axes = plt.subplots(1, 2, figsize=(13.8, 5.0))
for method_name in method_order:
    df_method = distance_logit_df[distance_logit_df['method'] == method_name]
    if df_method.empty:
        continue
    axes[0].scatter(
        df_method['teacher_embedding_distance'].to_numpy(dtype=float),
        df_method['logit_gap_abs'].to_numpy(dtype=float),
        s=8,
        alpha=0.10,
        color=plot_color_map[method_name],
        label=display_name_map[method_name],
    )
axes[0].set_xlabel('Teacher embedding distance to offline target')
axes[0].set_ylabel('|model logit - teacher offline logit|')
axes[0].set_title(f'Per-sample scatter | seed={int(TARGET_REPEAT_SEED)} | split={TARGET_SPLIT}')
axes[0].grid(True, alpha=0.25)
axes[0].legend(fontsize=9)

for method_name in method_order:
    curve_df = binned_curves.get(method_name, pd.DataFrame())
    if curve_df.empty:
        continue
    axes[1].plot(
        curve_df['distance_bin_mean'].to_numpy(dtype=float),
        curve_df['abs_gap_mean'].to_numpy(dtype=float),
        marker='o',
        ms=4,
        lw=2.0,
        color=plot_color_map[method_name],
        label=display_name_map[method_name],
    )
axes[1].set_xlabel('Teacher embedding distance (bin mean)')
axes[1].set_ylabel('Mean |model logit - teacher offline logit|')
axes[1].set_title('Binned trend of distance vs abs logit gap')
axes[1].grid(True, alpha=0.25)
axes[1].legend(fontsize=9)

fig.tight_layout()
fig_out = os.path.join(FIG_DIR, f'teacher_embedding_vs_logit_gap_seed_{int(TARGET_REPEAT_SEED)}_{safe_split}.png')
plt.savefig(fig_out, dpi=160, bbox_inches='tight')
print('Saved figure:', fig_out)
plt.show()